# Lab 04 — Serve a classifier, not just a string

Put the Topic 03 support-ticket task behind vLLM. Three skills: reserve the rendered context; distinguish complete, valid and correct predictions; diagnose a fresh service with explicit generation settings and positive/negative probes.

Implement **two functions**: a positional budget guard and classifier scoring. Streaming decoding, plotting and cleanup are supplied. Produce a confusion matrix and a justified configuration, not a greeting screenshot. Read [Book 04](https://levshaazz.github.io/llm-serving-mastery/en/book/04/#core-investigation).

A–C run offline on CPU. D compares two configurations on calibration, stops the server, then you inspect the results and freeze your choice. E opens held-out only after that decision, restarts the cached model and tests the selected setting. These are task-quality checks, not a latency A/B.

GPU defaults false. D–E require compatible idle Linux/WSL CUDA and the course's **24 GiB available-RAM safety gate**, not a universal vLLM minimum. Standard-RAM Colab is not certified; arrange an approved host if preflight fails. This v3 code passed separate instructor reference runs on T4 / Colab High RAM (8 October 2026): baseline versus prompt and baseline versus format on calibration; each froze baseline before held-out and stopped both owned servers. The format challenger was exercised on calibration, not held-out. Independent learner completion remains untested. Required v3 service artifact and grading thresholds are unchanged.


## A1. One task across Topics 03–04

Labels: billing (payments), access (login/permissions), bug (software malfunction **after successful login**). These are handwritten teaching tickets, not production data. The exact system prompt is printed in the provided cell.

Topic 03 compared Transformers FP16/NF4. Here FP16 is reloaded as a controlled serving baseline: we are **not** deploying the bitsandbytes NF4 result automatically. Repository ID locates weights, revision pins weights/tokenizer, served name topic04 selects the API model. Predict the cheapest wrong-name check and the next check for useful classification.


In [ ]:
IDENTITY_PREDICTION = ""
MODEL_ID='Qwen/Qwen2.5-0.5B-Instruct'
MODEL_REVISION='7ae557604adf67be50417f59c2c2f167def9a775'
SERVED_NAME='topic04'


### Provided CPU machinery

Run unchanged: no network, download or GPU import. Readable helpers are in the ZIP. Accept exactly one lowercase label after removing outer whitespace; never extract billing from billing because…. The supplied strict parser supports one text choice, not tools/reconnection.


In [ ]:
#@title Provided CPU machinery — run unchanged { display-mode: "form" }
parser_namespace={'__name__':'topic04_cpu_parser'}
exec("\"\"\"Bounded UTF-8/SSE decoder + single-choice chat lifecycle, standard library only.\n\nFraming follows WHATWG event streams (CR/LF/CRLF, comments, multiline data).\nChat completion is a separate, deliberately strict course contract, not all SSE.\n\"\"\"\nimport codecs\nimport json\nimport math\n\n\nclass SSEProtocolError(ValueError):\n    pass\n\n\nclass SSEDecoder:\n    def __init__(self, max_bytes=262144, max_events=4096):\n        self.decoder = codecs.getincrementaldecoder('utf-8-sig')('strict')\n        self.line = ''; self.data = []; self.event = ''; self.after_cr = False\n        self.bytes = 0; self.count = 0; self.max_bytes = max_bytes; self.max_events = max_events\n        self.closed = False\n\n    def _line(self):\n        line, self.line = self.line, ''\n        if not line:\n            if not self.data:\n                self.event = ''; return None\n            event = dict(event=self.event or 'message', data='\\n'.join(self.data))\n            self.data = []; self.event = ''; self.count += 1\n            if self.count > self.max_events: raise SSEProtocolError('event limit exceeded')\n            return event\n        if line.startswith(':'): return None\n        field, sep, value = line.partition(':')\n        if value.startswith(' '): value = value[1:]\n        if field == 'data': self.data.append(value)\n        elif field == 'event': self.event = value\n        # id/retry/unknown fields do not alter this finite HTTP replay client.\n        return None\n\n    def feed(self, raw):\n        if self.closed: raise SSEProtocolError('decoder already closed')\n        if not isinstance(raw,bytes): raise TypeError('feed requires bytes, not decoded strings')\n        self.bytes += len(raw)\n        if self.bytes > self.max_bytes: raise SSEProtocolError('byte limit exceeded')\n        text = self.decoder.decode(raw)\n        events = []\n        for char in text:\n            if self.after_cr:\n                self.after_cr = False\n                if char == '\\n': continue\n            if char in '\\r\\n':\n                event = self._line()\n                if event is not None: events.append(event)\n                self.after_cr = char == '\\r'\n            else: self.line += char\n        return events\n\n    def finish(self):\n        self.decoder.decode(b'',final=True)  # Incomplete UTF-8 is a protocol error.\n        self.closed = True\n        # WHATWG does not dispatch an unfinished event at EOF.\n        return bool(self.line or self.data)\n\n\nclass ChatStream:\n    \"\"\"No reconnect/retry: partial text is evidence, never a completed answer.\"\"\"\n    def __init__(self):\n        self.decoder = SSEDecoder(); self.events = []; self.content = ''\n        self.finish_reason = None; self.usage = None; self.done = False\n        self.first_event_s = None; self.first_content_s = None; self.last_s = 0.\n        self.error = None; self.closed = False; self.cancelled = False; self.unfinished_frame = False\n\n    def _event(self,event,at_s):\n        if self.done: raise SSEProtocolError('data after DONE')\n        if event['event'] != 'message': raise SSEProtocolError('only message events supported')\n        if self.first_event_s is None: self.first_event_s = at_s\n        body=event['data']\n        row=dict(index=len(self.events),at_s=at_s,event=event['event'],data=body)\n        self.events.append(row)\n        if body == '[DONE]':\n            if self.finish_reason is None: raise SSEProtocolError('DONE without finish_reason')\n            self.done = True; row['kind']='done'; return\n        obj=json.loads(body)\n        if not isinstance(obj,dict): raise SSEProtocolError('JSON object required')\n        if 'error' in obj: raise SSEProtocolError('server error event')\n        choices=obj.get('choices')\n        if not isinstance(choices,list): raise SSEProtocolError('choices list required')\n        usage=obj.get('usage')\n        if usage is not None:\n            if not isinstance(usage,dict): raise SSEProtocolError('invalid usage')\n            for key in ('prompt_tokens','completion_tokens','total_tokens'):\n                if type(usage.get(key)) is not int or usage[key]<0: raise SSEProtocolError('invalid usage counts')\n            if usage['total_tokens'] != usage['prompt_tokens']+usage['completion_tokens']:\n                raise SSEProtocolError('usage total mismatch')\n            self.usage=usage\n        if not choices:\n            if usage is None: raise SSEProtocolError('empty choices without usage')\n            row['kind']='usage'; return\n        if len(choices)!=1 or not isinstance(choices[0],dict) or type(choices[0].get('index')) is not int or choices[0]['index']!=0:\n            raise SSEProtocolError('only object choice zero supported')\n        choice=choices[0]; delta=choice.get('delta',{})\n        if not isinstance(delta,dict): raise SSEProtocolError('delta object required')\n        if any(key not in ('role','content') and value is not None for key,value in delta.items()):\n            raise SSEProtocolError('unsupported non-text delta')\n        if delta.get('role') not in (None,'assistant'): raise SSEProtocolError('unsupported role')\n        text=delta.get('content')\n        if text is not None and not isinstance(text,str): raise SSEProtocolError('content must be text or null')\n        if text:\n            if self.finish_reason is not None: raise SSEProtocolError('content after finish')\n            if self.first_content_s is None: self.first_content_s=at_s\n            self.content += text\n        finish=choice.get('finish_reason')\n        if finish is not None:\n            if finish not in ('stop','length') or self.finish_reason is not None:\n                raise SSEProtocolError('invalid or duplicate finish')\n            self.finish_reason=finish\n        row['kind']='finish' if finish is not None else ('content' if text else 'role_or_empty')\n\n    def feed(self,raw,at_s):\n        try:\n            if self.closed: raise SSEProtocolError('stream already closed')\n            if self.error: raise SSEProtocolError('stream already failed')\n            if type(at_s) not in (int,float) or not math.isfinite(at_s) or at_s<self.last_s:\n                raise SSEProtocolError('invalid or nonmonotonic arrival clock')\n            self.last_s=at_s\n            for event in self.decoder.feed(raw): self._event(event,at_s)\n        except (ValueError,TypeError,UnicodeError) as exc:\n            self.error=type(exc).__name__ + ': ' + str(exc)\n            raise\n\n    def finish(self,cancelled=False,transport_error=None):\n        if self.closed: raise SSEProtocolError('stream already closed')\n        self.closed=True; self.cancelled=cancelled\n        try: self.unfinished_frame=self.decoder.finish()\n        except UnicodeError as exc: self.error=type(exc).__name__ + ': incomplete UTF-8'\n        if transport_error: self.error=str(transport_error)\n        if self.error: status='failed'\n        elif cancelled: status='cancelled'\n        elif self.done and self.finish_reason is not None and not self.unfinished_frame: status='complete'\n        else: status='partial'\n        return dict(status=status,content=self.content,finish_reason=self.finish_reason,done=self.done,\n                    usage=self.usage,first_event_s=self.first_event_s,first_content_s=self.first_content_s,\n                    wall_s=self.last_s,byte_count=self.decoder.bytes,event_count=len(self.events),\n                    events=self.events,error=self.error,unfinished_frame=self.unfinished_frame,\n                    clock='client chunk-arrival wall; not token emission or GPU time')\n\n\ndef synthetic_fixture():\n    def chunk(delta,finish=None):\n        return dict(id='illustrative',object='chat.completion.chunk',model='topic04',\n                    choices=[dict(index=0,delta=delta,finish_reason=finish)])\n    payload=dict(model='topic04',messages=[dict(role='user',content='Say café ☕ briefly.')],\n                 max_tokens=16,temperature=0,stream=True,stream_options=dict(include_usage=True))\n    bodies=[chunk(dict(role='assistant',content='')),chunk(dict(content='Café ')),\n            chunk(dict(content='☕.')),chunk({},'stop'),\n            dict(choices=[],usage=dict(prompt_tokens=13,completion_tokens=4,total_tokens=17))]\n    # The synthetic usage counts are invented, not tokenizer measurements.\n    frames=[': heartbeat\\r\\n\\r\\n']+['data: '+json.dumps(b,ensure_ascii=False,separators=(',',':'))+'\\r\\n\\r\\n' for b in bodies]+['data: [DONE]\\r\\n\\r\\n']\n    arrival=[.01,.03,.10,.14,.15,.16,.17]\n    chunks=[]\n    for text,at_s in zip(frames,arrival):\n        raw=text.encode('utf-8')\n        # Always split CRLF; split UTF-8 continuation sequences when present.\n        cuts=sorted(set([len(raw)-3]+[i+1 for i,b in enumerate(raw) if b in (0xc3,0xe2)]))\n        previous=0\n        for end in cuts+[len(raw)]:\n            chunks.append(dict(at_s=at_s,hex=raw[previous:end].hex())); previous=end\n    stream=ChatStream()\n    for c in chunks: stream.feed(bytes.fromhex(c['hex']),c['at_s'])\n    complete=stream.finish()\n    prefix=ChatStream()\n    for c in chunks:\n        if c['at_s']>.10: break\n        prefix.feed(bytes.fromhex(c['hex']),c['at_s'])\n    partial=prefix.finish()\n    cancelled=ChatStream()\n    for c in chunks:\n        if c['at_s']>.10: break\n        cancelled.feed(bytes.fromhex(c['hex']),c['at_s'])\n    return dict(schema='topic04-sse-fixture-v1',kind='illustrative bytes, clocks and usage; no server or tokenizer executed',\n                request=payload,frames=frames,chunks=chunks,complete=complete,partial=partial,\n                cancelled=cancelled.finish(cancelled=True),source='https://html.spec.whatwg.org/multipage/server-sent-events.html')\n",parser_namespace)
exercise_namespace={'__name__':'topic04_cpu_exercises','parser_namespace':parser_namespace}
exec("\"\"\"Provided CPU infrastructure, exercises and checks. No student solutions.\n\nFinite, single-choice, text-only chat is the teaching scope. A full OpenAI client,\ntool calls, reconnection and production timeout enforcement are not implemented here.\n\"\"\"\nimport json\nSSEDecoder = parser_namespace[\"SSEDecoder\"]\n\n\ndef budget_inputs(prompt_tokens, output_cap, context_limit):\n    \"\"\"Validate counts, separately from the student's admission calculation.\"\"\"\n    for name, value in ((\"prompt_tokens\", prompt_tokens), (\"output_cap\", output_cap),\n                        (\"context_limit\", context_limit)):\n        if type(value) is not int or value < 0:\n            raise ValueError(name + \" must be a nonnegative integer\")\n    if context_limit == 0:\n        raise ValueError(\"context_limit must be positive\")\n    if output_cap == 0:\n        raise ValueError(\"output_cap must be positive for this vLLM generation request\")\n\n\ndef decode_chunks(chunks):\n    \"\"\"Incremental UTF-8 + SSE framing; leave lifecycle decisions to the student.\n\n    Arrival clocks are intentionally omitted: resegmentation keeps bytes, not timings.\n    Only the finite teaching subset (JSON data or [DONE]) is returned.\n    \"\"\"\n    decoder = SSEDecoder()\n    records = []\n    for raw in chunks:\n        for frame in decoder.feed(raw):\n            if frame['event'] != 'message':\n                raise ValueError('Only finite message events supported')\n            data = frame['data']\n            records.append(\"[DONE]\" if data == \"[DONE]\" else json.loads(data))\n    if decoder.finish():\n        raise ValueError('Unfinished SSE frame at EOF; do not call it complete')\n    return records\n\n\ndef check_admission(admit):\n    cases = [(33, 16, 2048, True), (1800, 512, 2048, False),\n             (1800, 128, 2048, True), (2329, 16, 2048, False),\n             (2032, 16, 2048, True), (2033, 16, 2048, False)]\n    for p, o, limit, expected in cases:\n        actual = admit(p, o, limit)\n        assert type(actual) is bool and actual is expected, (p, o, limit, actual)\n    for args in [(True, 1, 2048), (-1, 1, 2048), (1, 1.5, 2048), (1, 1, 0), (1, 0, 2048)]:\n        try:\n            admit(*args)\n        except ValueError:\n            pass\n        else:\n            raise AssertionError(\"Invalid budget input accepted: \" + repr(args))\n    return len(cases) + 5\n\n\ndef event(content=None, finish=None):\n    delta = {} if content is None else {\"content\": content}\n    return {\"choices\": [{\"index\": 0, \"delta\": delta, \"finish_reason\": finish}]}\n\n\ndef check_reducer(reduce_stream):\n    \"\"\"Supported envelopes; the exercise owns concatenation and success policy.\n\n    Malformed JSON/UTF-8 and the broader server protocol are the provided parser's\n    separate responsibility. Duplicate/out-of-order lifecycle data must fail.\n    \"\"\"\n    prefix = [{\"choices\": [{\"index\": 0, \"delta\": {\"role\": \"assistant\"},\n                             \"finish_reason\": None}]}, event(\"Hello\"), event(\" world\")]\n    usage = {\"choices\": [], \"usage\": {\"prompt_tokens\": 33,\n              \"completion_tokens\": 10, \"total_tokens\": 43}}\n    cases = [\n        (prefix + [event(finish=\"stop\"), usage, \"[DONE]\"], \"Hello world\", \"complete\"),\n        (prefix, \"Hello world\", \"partial\"),\n        (prefix + [event(finish=\"stop\")], \"Hello world\", \"partial\"),\n        (prefix + [\"[DONE]\"], \"Hello world\", \"failed\"),\n        (prefix + [{\"error\": {\"message\": \"engine failed\"}}, event(\" late\"), \"[DONE]\"], \"Hello world\", \"failed\"),\n        (prefix + [event(finish=\"length\"), \"[DONE]\"], \"Hello world\", \"complete\"),\n        (prefix + [event(finish=\"stop\"), event(\" late\"), \"[DONE]\"], \"Hello world\", \"failed\"),\n        (prefix + [event(finish=\"stop\"), \"[DONE]\", event(\" late\")], \"Hello world\", \"failed\"),\n        (prefix + [event(finish=\"stop\"), event(finish=\"stop\"), \"[DONE]\"], \"Hello world\", \"failed\"),\n    ]\n    for records, content, status in cases:\n        result = reduce_stream(records)\n        assert result[\"content\"] == content, (records, result)\n        assert result[\"status\"] == status, (records, result)\n        assert type(result[\"done\"]) is bool and result[\"done\"] is (\"[DONE]\" in records), (records, result)\n        reasons = [c.get(\"finish_reason\") for r in records if isinstance(r, dict)\n                   for c in r.get(\"choices\", []) if c.get(\"finish_reason\") is not None]\n        assert result[\"finish_reason\"] == (reasons[-1] if reasons else None), (records, result)\n    return len(cases)\n\n\ndef byte_partitions(wire):\n    \"\"\"Same byte transcript, three transport segmentations; no invented network run.\"\"\"\n    return {\"one_read\": [wire], \"one_byte\": [bytes([v]) for v in wire],\n            \"seventeen_bytes\": [wire[i:i+17] for i in range(0, len(wire), 17)]}\n",exercise_namespace)
task_namespace={'__name__':'topic04_ml_task','parser_namespace':parser_namespace}
exec("\"\"\"Provided ML task machinery. No network/GPU work at import, no student answer.\"\"\"\nimport math\nChatStream = parser_namespace[\"ChatStream\"]\n\nLABELS = ('billing', 'access', 'bug')\nSYSTEM = ('Classify the support ticket. Reply with exactly one lowercase label: billing, access, or bug. '\n          'billing = payments, invoices, refunds; access = login or account permissions; '\n          'bug = software malfunction after successful login. No explanation.')\n\n\ndef sampling_distribution(logits=(2., 1., 0.), temperature=1., top_p=1.):\n    \"\"\"Toy next-token distribution; NOT measured Qwen logits/class probabilities.\"\"\"\n    if not math.isfinite(temperature) or temperature < 0 or not 0 < top_p <= 1 or not logits or not all(math.isfinite(v) for v in logits):\n        raise ValueError('Finite logits, nonnegative temperature and 0 < top_p <= 1 required')\n    if temperature == 0:\n        best = max(range(len(logits)), key=lambda i: logits[i])\n        return [float(i == best) for i in range(len(logits))]\n    values = [math.exp((v-max(logits))/temperature) for v in logits]\n    probs = [v/sum(values) for v in values]\n    order = sorted(range(len(probs)), key=lambda i: probs[i], reverse=True)\n    kept, mass = [], 0.\n    for i in order:\n        kept.append(i); mass += probs[i]\n        if mass >= top_p: break\n    return [probs[i]/mass if i in kept else 0. for i in range(len(probs))]\n\n\ndef replay_response(records):\n    \"\"\"Use the tested strict wire parser rather than ask ML students to write it.\"\"\"\n    import json\n    parser = ChatStream()\n    for record in records:\n        value = record if record == '[DONE]' else json.dumps(record, ensure_ascii=False)\n        parser.feed(('data: '+value+'\\n\\n').encode('utf-8'), 0.)\n    state = parser.finish()\n    return {k: state[k] for k in ('content', 'finish_reason', 'done', 'status')}\n\n\ndef prediction_row(case_id, expected, content, status='complete', finish_reason='stop'):\n    if expected not in LABELS or status not in ('complete', 'partial', 'failed') or not isinstance(content, str):\n        raise ValueError('Invalid task observation')\n    # Exact lowercase label after outer whitespace only; no substring extraction.\n    candidate = content.strip()\n    predicted = candidate if status == 'complete' and candidate in LABELS else None\n    return dict(case_id=case_id, expected=expected, content=content, status=status,\n                finish_reason=finish_reason, predicted=predicted)\n\n\ndef evaluation_fixture():\n    \"\"\"Eight invented attempts, covering wrong label, invalid format and failures.\"\"\"\n    return [prediction_row('e1','billing','billing'), prediction_row('e2','billing','access'),\n            prediction_row('e3','access','access'), prediction_row('e4','bug','bug',finish_reason='length'),\n            prediction_row('e5','bug','billing because payment failed'),\n            prediction_row('e6','access','access',status='partial'),\n            prediction_row('e7','billing','',status='failed'), prediction_row('e8','bug','bug')]\n\n\ndef check_scores(score):\n    fixtures = [([], (0,0,0,0)), (evaluation_fixture(), (8,4,5,6)),\n                ([prediction_row('x','bug','access')], (1,0,1,1)),\n                ([prediction_row('x','access','access',status='partial')], (1,0,0,0)),\n                ([prediction_row('x','bug','bug',finish_reason='length')], (1,1,1,1)),\n                ([prediction_row('x','billing','Billing')], (1,0,0,1))]\n    for rows, counts in fixtures:\n        result = score(rows)\n        for key, count in zip(('attempts','correct','valid','complete'), counts):\n            assert type(result[key]) is int and result[key] == count, (key,rows,result)\n        n, correct, valid, complete = counts\n        for key, expected in [('accuracy',correct/n if n else None),('valid_rate',valid/n if n else None),\n                              ('completion_rate',complete/n if n else None)]:\n            assert result[key] == expected, (key,rows,result)\n        matrix = result['confusion']\n        assert len(matrix)==3 and all(len(row)==4 for row in matrix)\n        assert all(type(v) is int and v>=0 for row in matrix for v in row)\n        assert sum(map(sum,matrix))==n\n        for i,label in enumerate(LABELS):\n            expected_row = [sum(r['expected']==label and r['predicted']==p for r in rows)\n                            for p in (*LABELS,None)]\n            assert matrix[i]==expected_row, (label,matrix,expected_row)\n    return len(fixtures)\n\n\ndef live_task_probe(client, tokenizer, cases, output_cap=8, structured=False, save_partial=None, system=SYSTEM):\n    \"\"\"One frozen setting, sequential task probe inside the owned service lifetime.\n\n    The caller's absolute 120s bound covers the WHOLE auxiliary probe. Never retry.\n    Failures remain rows; accuracy denominators must include every attempted ticket.\n    \"\"\"\n    if type(output_cap) is not int or not 1 <= output_cap <= 16 or type(structured) is not bool:\n        raise ValueError('Task cap must be 1..16; structured must be bool')\n    if not isinstance(system,str) or not system.strip():\n        raise ValueError('Explicit nonempty system instruction required')\n    observations = []\n    if len({c['id'] for c in cases}) != len(cases) or any(c['label'] not in LABELS for c in cases):\n        raise ValueError('Unique case IDs and known gold labels required')\n    def record(status):\n        return dict(contract='topic04-task-probe-v1',status=status,temperature=0,\n                    output_cap=output_cap,structured=structured,system=system,observations=observations)\n    for case in cases:\n        messages=[dict(role='system',content=system),dict(role='user',content=case['text'])]\n        encoded=tokenizer.apply_chat_template(messages,tokenize=True,add_generation_prompt=True)\n        ids=encoded['input_ids'] if hasattr(encoded,'keys') else encoded\n        if ids and isinstance(ids[0],list): ids=ids[0]\n        if len(ids)+output_cap>2048: raise ValueError('Task request exceeds frozen context budget')\n        request=dict(model='topic04',messages=messages,temperature=0,max_tokens=output_cap)\n        if structured: request['extra_body']={'structured_outputs':{'choice':list(LABELS)}}\n        try:\n            answer=client.chat.completions.create(**request)\n            choice=answer.choices[0]\n            state='complete' if choice.finish_reason in ('stop','length') else 'failed'\n            row=prediction_row(case['id'],case['label'],choice.message.content or '',\n                               status=state,finish_reason=choice.finish_reason)\n            row['usage']=answer.usage.model_dump() if answer.usage else None\n        except TimeoutError:\n            # Save the failed attempt, but never suppress the outer absolute deadline.\n            row=prediction_row(case['id'],case['label'],'',status='failed')\n            row.update(error_type='TimeoutError',request=request,rendered_input_ids=ids)\n            observations.append(row)\n            if save_partial is not None: save_partial(record('deadline-exceeded'))\n            raise\n        except Exception as exc:\n            row=prediction_row(case['id'],case['label'],'',status='failed')\n            row['error_type']=type(exc).__name__\n        row.update(request=request,rendered_input_ids=ids)\n        observations.append(row)\n        if save_partial is not None: save_partial(record('running'))\n    return record('complete')\n\n\ndef comparison_policies(challenger='prompt', output_cap=8):\n    \"\"\"A versus B changes ONE factor; declarations precede measurements.\"\"\"\n    if challenger not in ('prompt', 'format') or type(output_cap) is not int or not 1 <= output_cap <= 16:\n        raise ValueError('Choose prompt or format, with integer cap 1..16')\n    clarified = SYSTEM + (' Decide by the problem that needs fixing. If login succeeds but a feature '\n        'crashes, loses data or produces a wrong result, choose bug. Merely mentioning login '\n        'does not mean access. Choose access when signing in or obtaining permission fails.')\n    return [dict(name='baseline', output_cap=output_cap, structured=False, system=SYSTEM),\n            dict(name=challenger, output_cap=output_cap, structured=challenger=='format',\n                 system=clarified if challenger=='prompt' else SYSTEM)]\n\n\ndef baseline_and_recall(rows, score):\n    \"\"\"Provided statistics, not a third implementation task.\"\"\"\n    n = len(rows)\n    return dict(constant_access_correct=sum(r['expected']=='access' for r in rows),\n                constant_access_accuracy=sum(r['expected']=='access' for r in rows)/n if n else None,\n                recall={label: (sum(r['expected']==label and r['predicted']==label for r in rows)\n                                /sum(r['expected']==label for r in rows)\n                                if any(r['expected']==label for r in rows) else None) for label in LABELS},\n                scores=score(rows))\n\n\ndef freeze_selection(comparison, policies, chosen, explanation, path):\n    \"\"\"Save a student choice from measured calibration BEFORE opening test data.\"\"\"\n    import hashlib, json\n    from pathlib import Path\n    if comparison.get('split') != 'calibration' or comparison.get('status') != 'complete':\n        raise ValueError('Completed calibration comparison required')\n    names = [p['name'] for p in policies]\n    if chosen not in names or not isinstance(explanation,str) or not explanation.strip():\n        raise ValueError('Choose an observed policy and explain the calibration trade-off')\n    if [r['policy'] for r in comparison['results']] != policies:\n        raise ValueError('Policy changed after calibration')\n    if len(policies)!=2 or policies!=comparison_policies(policies[1]['name'],policies[0]['output_cap']):\n        raise ValueError('Declare exactly the matched baseline and one-factor challenger')\n    for result in comparison['results']:\n        policy,probe=result['policy'],result['probe']\n        if probe.get('status')!='complete' or probe.get('temperature')!=0:\n            raise ValueError('Incomplete or unmatched calibration probe')\n        if any(probe.get(k)!=policy[k] for k in ('system','structured','output_cap')):\n            raise ValueError('Observed probe settings differ from the declared policy')\n        for row in probe['observations']:\n            request=row['request']\n            constraint={'structured_outputs':{'choice':list(LABELS)}} if policy['structured'] else None\n            if (request.get('temperature')!=0 or request.get('max_tokens')!=policy['output_cap']\n                    or request.get('model')!='topic04' or request.get('extra_body')!=constraint\n                    or request['messages'][0]!=dict(role='system',content=policy['system'])):\n                raise ValueError('A calibration request violates its declared control')\n    populations = [[(r['case_id'],r['expected'],r['request']['messages'][-1]['content'])\n                    for r in result['probe']['observations']] for result in comparison['results']]\n    if len(populations)!=2 or not populations[0] or populations[0]!=populations[1]:\n        raise ValueError('Both candidates must retain the same calibration attempts in order')\n    payload=dict(contract='topic04-frozen-choice-v1', selected=policies[names.index(chosen)],\n                 explanation=explanation.strip(), comparison=comparison)\n    digest=hashlib.sha256(json.dumps(payload,sort_keys=True,ensure_ascii=False).encode()).hexdigest()\n    record=dict(payload=payload,sha256=digest)\n    target=Path(path);target.parent.mkdir(parents=True,exist_ok=True)\n    # An old decision cannot silently become a new one after test results.\n    if target.exists(): raise FileExistsError('New choice requires a new attempt directory')\n    target.write_text(json.dumps(record,indent=2,ensure_ascii=False))\n    return record\n\n\ndef read_frozen(path):\n    import hashlib, json\n    from pathlib import Path\n    record=json.loads(Path(path).read_text())\n    digest=hashlib.sha256(json.dumps(record['payload'],sort_keys=True,ensure_ascii=False).encode()).hexdigest()\n    if record['sha256']!=digest or record['payload']['contract']!='topic04-frozen-choice-v1':\n        raise ValueError('Frozen decision changed')\n    return record\n\n\ndef heldout_cases(frozen):\n    \"\"\"Default access after freeze; PUBLIC teaching data, not cryptographic secrecy.\"\"\"\n    import hashlib, json\n    digest=hashlib.sha256(json.dumps(frozen['payload'],sort_keys=True,ensure_ascii=False).encode()).hexdigest()\n    if frozen['sha256']!=digest or frozen['payload']['contract']!='topic04-frozen-choice-v1':\n        raise ValueError('Valid frozen decision required before held-out access')\n    return [dict(id='s1',label='billing',text='The receipt for our payment has the wrong tax number.'),\n            dict(id='s2',label='billing',text='A cancelled paid plan has still taken money from my card.'),\n            dict(id='s3',label='access',text='The password reset link expired, so I cannot sign in.'),\n            dict(id='s4',label='access',text='The administrator has not granted me permission to view the project.'),\n            dict(id='s5',label='bug',text='Login works, but the application freezes whenever I sort the table.'),\n            dict(id='s6',label='bug',text='After signing in, every exported PDF has missing pages.')]\n",task_namespace)
budget_inputs=exercise_namespace['budget_inputs']
check_admission=exercise_namespace['check_admission']
decode_chunks=exercise_namespace['decode_chunks']
byte_partitions=exercise_namespace['byte_partitions']
assemble_response=task_namespace['replay_response']  # Provided, not mandatory student code.
prediction_row=task_namespace['prediction_row']
check_scores=task_namespace['check_scores']
LABELS=task_namespace['LABELS']
plots_namespace={'__name__':'topic04_plots'}
exec("\"\"\"Readable task figures; no network or GPU work at import.\"\"\"\n\ndef annotation_color(rgba):\n    # Relative luminance, with sRGB linearization; choose higher text contrast.\n    values=[v/12.92 if v<=.04045 else ((v+.055)/1.055)**2.4 for v in rgba[:3]]\n    light=sum(w*v for w,v in zip((.2126,.7152,.0722),values))\n    return 'black' if (light+.05)/.05 >= 1.05/(light+.05) else 'white'\n\n\ndef plot_matrix(rows, score, title):\n    import matplotlib.pyplot as plt\n    import numpy as np\n    labels=('billing','access','bug')\n    matrix=np.array(score(rows)['confusion'])\n    fig,ax=plt.subplots(figsize=(8,4.4),layout='constrained')\n    im=ax.imshow(matrix,cmap='Blues',vmin=0,vmax=max(1,matrix.max()))\n    for i in range(3):\n        for j in range(4):\n            ax.text(j,i,str(matrix[i,j]),ha='center',va='center',fontsize=13,\n                    color=annotation_color(im.cmap(im.norm(matrix[i,j]))))\n    ax.set(xticks=range(4),xticklabels=[*labels,'invalid /\\nincomplete'],yticks=range(3),\n           yticklabels=labels,xlabel='Predicted label',ylabel='True label',title=title)\n    fig.colorbar(im,ax=ax,label='Attempt count',ticks=range(int(matrix.max())+1))\n    plt.show()\n    return fig\n\n\ndef plot_sampling(probs):\n    import matplotlib.pyplot as plt\n    fig,ax=plt.subplots(figsize=(7,3.4),layout='constrained')\n    ax.bar(['token A','token B','token C'],probs)\n    ax.set(ylim=(0,1),ylabel='Next-token probability',title='Toy logits [2,1,0], after T and top_p')\n    plt.show()\n    return fig\n",plots_namespace)
plot_matrix=plots_namespace['plot_matrix']
plot_sampling=plots_namespace['plot_sampling']
task_summary=lambda rows:task_namespace['baseline_and_recall'](rows,score_predictions)
print('Exact system instruction:',task_namespace['SYSTEM'])


## A2. Implement the positional budget

Count **rendered chat**, including system instructions and role markers, not words. Reserve prompt_tokens + output_cap against context_limit. 1800+512 exceeds 2048; 1800+128 fits; equality fits. A cap is a maximum, not a promised output length.

Call budget_inputs first: integers, not bool; positive output cap and context limit. Return a Python bool. This conservative application rule proves neither free KV memory nor classification quality.


In [ ]:
def admit_request(prompt_tokens, output_cap, context_limit):
    budget_inputs(prompt_tokens, output_cap, context_limit)
    # TODO: compare reserved positions with the limit.
    raise NotImplementedError('Implement the rendered context budget')


In [ ]:
ADMISSION_CHECKS=check_admission(admit_request)
assert ADMISSION_CHECKS==11
print(ADMISSION_CHECKS,'admission checks passed')


## B1. Logits and generation settings

Qwen maps the last hidden state (width 896) to **151936 vocabulary logits**, not three class scores. Select the next token, append it, run the next step. A class name may need several tokens. EOS or a cap stops generation.

Invented logits [2,1,0]: for positive T compute softmax(z/T). T=0 means argmax, not division by zero. top_p keeps the smallest descending prefix reaching p, then renormalizes. **Next-token probability is not calibrated class confidence.** Change temperature/top_p and predict the retained tokens first.


In [ ]:
TEMPERATURE=1.0  # Compare 0.5, 2.0, 0.0.
TOP_P=0.8       # Compare 1.0 and 0.8 at the SAME temperature.
TOY_PROBS=task_namespace['sampling_distribution'](temperature=TEMPERATURE,top_p=TOP_P)
assert abs(sum(TOY_PROBS)-1)<1e-12
print('Toy token A/B/C:',TOY_PROBS)
SAMPLING_PREDICTION = ""


## B2. Score ALL attempted classifications

Rows have expected, predicted and status. The provided parser sets predicted=None for a non-label or incomplete/failed response, even a partial prefix saying access. A complete length-limited answer exactly bug is valid; retain its finish reason.

Return integer attempts/correct/valid/complete and accuracy/valid_rate/completion_rate. All rates divide by **all attempts**. With zero attempts return None rates. Return a **3×4 confusion matrix**: true rows billing/access/bug; predicted columns billing/access/bug/invalid-or-incomplete. Every attempt occupies one cell.

The invented eight-case fixture has four correct, five valid and six complete: 4/8, 5/8, 6/8. The tempting 4/5 is conditional accuracy, not service accuracy. Implement the counts, matrix and rates.


In [ ]:
def score_predictions(rows):
    # TODO: count all attempts; construct the 3x4 matrix in LABELS order.
    # Keys: attempts, correct, valid, complete, accuracy, valid_rate,
    # completion_rate, confusion. Keep predicted=None rows.
    raise NotImplementedError('Implement service-task evaluation')


In [ ]:
SCORING_CHECKS=check_scores(score_predictions)
assert SCORING_CHECKS==6
FIXTURE_ROWS=task_namespace['evaluation_fixture']()
for row in FIXTURE_ROWS: print(row)
print(score_predictions(FIXTURE_ROWS))


## C1. Inspect actual Qwen errors

These **historical Transformers FP16/T4** outputs were measured in Topic 03, not vLLM. Each ticket retains text, gold label, rendered chat and input/output IDs. All 12 are valid labels; only six are correct. In g2 the Save button malfunctions after successful login, but the model says access.

Compute the matrix and read two wrong predictions in full. Predict the constant-access baseline before computing it: four of twelve gold labels are access, so it scores 4/12. The model scores 6/12 but recall for billing is zero. Propose two competing explanations, then a test that distinguishes them. Twelve teaching cases cannot establish production accuracy.


In [ ]:
#@title Load dated public task observations — run unchanged { display-mode: "form" }
import json
RECORDED_TASK=json.loads("{\"scope\":\"Historical Topic03 Transformers FP16 on T4; NOT a vLLM task result\",\"source\":\"seminars/runs/2026-10-06-topic03-v6-colab/gpu-capture.json\",\"source_sha256\":\"3f56cefbae500b5f437b547930303d3fbe57ccb1797616cf358de55c3e41902c\",\"cases\":[{\"id\":\"b1\",\"text\":\"I was charged twice for my monthly subscription. Please refund the duplicate.\",\"label\":\"billing\"},{\"id\":\"b2\",\"text\":\"Please send an invoice for the payment we made last week.\",\"label\":\"billing\"},{\"id\":\"b3\",\"text\":\"My credit card expired and the subscription payment was declined.\",\"label\":\"billing\"},{\"id\":\"b4\",\"text\":\"I cancelled the subscription but a new charge appeared on my bank statement.\",\"label\":\"billing\"},{\"id\":\"a1\",\"text\":\"I forgot my password and cannot sign in to my account.\",\"label\":\"access\"},{\"id\":\"a2\",\"text\":\"Our new employee needs permission to open the team workspace.\",\"label\":\"access\"},{\"id\":\"a3\",\"text\":\"I replaced my phone and no longer have my two-factor login codes.\",\"label\":\"access\"},{\"id\":\"a4\",\"text\":\"My account is locked after too many failed sign-in attempts.\",\"label\":\"access\"},{\"id\":\"g1\",\"text\":\"I can log in, but clicking Export crashes the application.\",\"label\":\"bug\"},{\"id\":\"g2\",\"text\":\"After successful login, the Save button does nothing and my edits disappear.\",\"label\":\"bug\"},{\"id\":\"g3\",\"text\":\"I am signed in. The search page shows an error for every search query.\",\"label\":\"bug\"},{\"id\":\"g4\",\"text\":\"My account works, but every image I upload is displayed upside down.\",\"label\":\"bug\"}],\"samples\":[{\"wall_s\":0.058112368999900355,\"output_tokens\":2,\"output_token_ids\":[2313,151645],\"output\":\"bug\",\"case_id\":\"b1\",\"expected_label\":\"billing\",\"rendered_chat\":\"<|im_start|>system\\nClassify the support ticket. Reply with exactly one lowercase label: billing, access, or bug. billing = payments, invoices, refunds; access = login or account permissions; bug = software malfunction after successful login. No explanation.<|im_end|>\\n<|im_start|>user\\nI was charged twice for my monthly subscription. Please refund the duplicate.<|im_end|>\\n<|im_start|>assistant\\n\",\"input_token_ids\":[151644,8948,198,1957,1437,279,1824,11727,13,17841,448,6896,825,42047,2383,25,33531,11,2615,11,476,9876,13,33531,284,14173,11,64052,11,72518,26,2615,284,5858,476,2692,8541,26,9876,284,3162,71187,1283,6849,5858,13,2308,16148,13,151645,198,151644,872,198,40,572,11430,10917,369,847,15088,15142,13,5209,20965,279,22513,13,151645,198,151644,77091,198],\"stop_reason\":\"eos\",\"predicted_label\":\"bug\",\"valid_label\":true,\"correct\":false},{\"wall_s\":0.05793165599993699,\"output_tokens\":2,\"output_token_ids\":[5211,151645],\"output\":\"access\",\"case_id\":\"b2\",\"expected_label\":\"billing\",\"rendered_chat\":\"<|im_start|>system\\nClassify the support ticket. Reply with exactly one lowercase label: billing, access, or bug. billing = payments, invoices, refunds; access = login or account permissions; bug = software malfunction after successful login. No explanation.<|im_end|>\\n<|im_start|>user\\nPlease send an invoice for the payment we made last week.<|im_end|>\\n<|im_start|>assistant\\n\",\"input_token_ids\":[151644,8948,198,1957,1437,279,1824,11727,13,17841,448,6896,825,42047,2383,25,33531,11,2615,11,476,9876,13,33531,284,14173,11,64052,11,72518,26,2615,284,5858,476,2692,8541,26,9876,284,3162,71187,1283,6849,5858,13,2308,16148,13,151645,198,151644,872,198,5501,3624,458,24615,369,279,8160,582,1865,1537,2003,13,151645,198,151644,77091,198],\"stop_reason\":\"eos\",\"predicted_label\":\"access\",\"valid_label\":true,\"correct\":false},{\"wall_s\":0.056659867999997005,\"output_tokens\":2,\"output_token_ids\":[5211,151645],\"output\":\"access\",\"case_id\":\"b3\",\"expected_label\":\"billing\",\"rendered_chat\":\"<|im_start|>system\\nClassify the support ticket. Reply with exactly one lowercase label: billing, access, or bug. billing = payments, invoices, refunds; access = login or account permissions; bug = software malfunction after successful login. No explanation.<|im_end|>\\n<|im_start|>user\\nMy credit card expired and the subscription payment was declined.<|im_end|>\\n<|im_start|>assistant\\n\",\"input_token_ids\":[151644,8948,198,1957,1437,279,1824,11727,13,17841,448,6896,825,42047,2383,25,33531,11,2615,11,476,9876,13,33531,284,14173,11,64052,11,72518,26,2615,284,5858,476,2692,8541,26,9876,284,3162,71187,1283,6849,5858,13,2308,16148,13,151645,198,151644,872,198,5050,6668,3701,26391,323,279,15142,8160,572,18758,13,151645,198,151644,77091,198],\"stop_reason\":\"eos\",\"predicted_label\":\"access\",\"valid_label\":true,\"correct\":false},{\"wall_s\":0.055597407999812276,\"output_tokens\":2,\"output_token_ids\":[5211,151645],\"output\":\"access\",\"case_id\":\"b4\",\"expected_label\":\"billing\",\"rendered_chat\":\"<|im_start|>system\\nClassify the support ticket. Reply with exactly one lowercase label: billing, access, or bug. billing = payments, invoices, refunds; access = login or account permissions; bug = software malfunction after successful login. No explanation.<|im_end|>\\n<|im_start|>user\\nI cancelled the subscription but a new charge appeared on my bank statement.<|im_end|>\\n<|im_start|>assistant\\n\",\"input_token_ids\":[151644,8948,198,1957,1437,279,1824,11727,13,17841,448,6896,825,42047,2383,25,33531,11,2615,11,476,9876,13,33531,284,14173,11,64052,11,72518,26,2615,284,5858,476,2692,8541,26,9876,284,3162,71187,1283,6849,5858,13,2308,16148,13,151645,198,151644,872,198,40,25681,279,15142,714,264,501,6757,9723,389,847,6073,5114,13,151645,198,151644,77091,198],\"stop_reason\":\"eos\",\"predicted_label\":\"access\",\"valid_label\":true,\"correct\":false},{\"wall_s\":0.06494631800001116,\"output_tokens\":2,\"output_token_ids\":[5211,151645],\"output\":\"access\",\"case_id\":\"a1\",\"expected_label\":\"access\",\"rendered_chat\":\"<|im_start|>system\\nClassify the support ticket. Reply with exactly one lowercase label: billing, access, or bug. billing = payments, invoices, refunds; access = login or account permissions; bug = software malfunction after successful login. No explanation.<|im_end|>\\n<|im_start|>user\\nI forgot my password and cannot sign in to my account.<|im_end|>\\n<|im_start|>assistant\\n\",\"input_token_ids\":[151644,8948,198,1957,1437,279,1824,11727,13,17841,448,6896,825,42047,2383,25,33531,11,2615,11,476,9876,13,33531,284,14173,11,64052,11,72518,26,2615,284,5858,476,2692,8541,26,9876,284,3162,71187,1283,6849,5858,13,2308,16148,13,151645,198,151644,872,198,40,28595,847,3552,323,4157,1841,304,311,847,2692,13,151645,198,151644,77091,198],\"stop_reason\":\"eos\",\"predicted_label\":\"access\",\"valid_label\":true,\"correct\":true},{\"wall_s\":0.05672998199997892,\"output_tokens\":2,\"output_token_ids\":[5211,151645],\"output\":\"access\",\"case_id\":\"a2\",\"expected_label\":\"access\",\"rendered_chat\":\"<|im_start|>system\\nClassify the support ticket. Reply with exactly one lowercase label: billing, access, or bug. billing = payments, invoices, refunds; access = login or account permissions; bug = software malfunction after successful login. No explanation.<|im_end|>\\n<|im_start|>user\\nOur new employee needs permission to open the team workspace.<|im_end|>\\n<|im_start|>assistant\\n\",\"input_token_ids\":[151644,8948,198,1957,1437,279,1824,11727,13,17841,448,6896,825,42047,2383,25,33531,11,2615,11,476,9876,13,33531,284,14173,11,64052,11,72518,26,2615,284,5858,476,2692,8541,26,9876,284,3162,71187,1283,6849,5858,13,2308,16148,13,151645,198,151644,872,198,7981,501,9364,3880,7882,311,1787,279,2083,27514,13,151645,198,151644,77091,198],\"stop_reason\":\"eos\",\"predicted_label\":\"access\",\"valid_label\":true,\"correct\":true},{\"wall_s\":0.054001018000008116,\"output_tokens\":2,\"output_token_ids\":[5211,151645],\"output\":\"access\",\"case_id\":\"a3\",\"expected_label\":\"access\",\"rendered_chat\":\"<|im_start|>system\\nClassify the support ticket. Reply with exactly one lowercase label: billing, access, or bug. billing = payments, invoices, refunds; access = login or account permissions; bug = software malfunction after successful login. No explanation.<|im_end|>\\n<|im_start|>user\\nI replaced my phone and no longer have my two-factor login codes.<|im_end|>\\n<|im_start|>assistant\\n\",\"input_token_ids\":[151644,8948,198,1957,1437,279,1824,11727,13,17841,448,6896,825,42047,2383,25,33531,11,2615,11,476,9876,13,33531,284,14173,11,64052,11,72518,26,2615,284,5858,476,2692,8541,26,9876,284,3162,71187,1283,6849,5858,13,2308,16148,13,151645,198,151644,872,198,40,12575,847,4540,323,902,5021,614,847,1378,74097,5858,13912,13,151645,198,151644,77091,198],\"stop_reason\":\"eos\",\"predicted_label\":\"access\",\"valid_label\":true,\"correct\":true},{\"wall_s\":0.058577301000013904,\"output_tokens\":2,\"output_token_ids\":[5211,151645],\"output\":\"access\",\"case_id\":\"a4\",\"expected_label\":\"access\",\"rendered_chat\":\"<|im_start|>system\\nClassify the support ticket. Reply with exactly one lowercase label: billing, access, or bug. billing = payments, invoices, refunds; access = login or account permissions; bug = software malfunction after successful login. No explanation.<|im_end|>\\n<|im_start|>user\\nMy account is locked after too many failed sign-in attempts.<|im_end|>\\n<|im_start|>assistant\\n\",\"input_token_ids\":[151644,8948,198,1957,1437,279,1824,11727,13,17841,448,6896,825,42047,2383,25,33531,11,2615,11,476,9876,13,33531,284,14173,11,64052,11,72518,26,2615,284,5858,476,2692,8541,26,9876,284,3162,71187,1283,6849,5858,13,2308,16148,13,151645,198,151644,872,198,5050,2692,374,16061,1283,2238,1657,4641,1841,3419,13553,13,151645,198,151644,77091,198],\"stop_reason\":\"eos\",\"predicted_label\":\"access\",\"valid_label\":true,\"correct\":true},{\"wall_s\":0.05656722300000183,\"output_tokens\":2,\"output_token_ids\":[2313,151645],\"output\":\"bug\",\"case_id\":\"g1\",\"expected_label\":\"bug\",\"rendered_chat\":\"<|im_start|>system\\nClassify the support ticket. Reply with exactly one lowercase label: billing, access, or bug. billing = payments, invoices, refunds; access = login or account permissions; bug = software malfunction after successful login. No explanation.<|im_end|>\\n<|im_start|>user\\nI can log in, but clicking Export crashes the application.<|im_end|>\\n<|im_start|>assistant\\n\",\"input_token_ids\":[151644,8948,198,1957,1437,279,1824,11727,13,17841,448,6896,825,42047,2383,25,33531,11,2615,11,476,9876,13,33531,284,14173,11,64052,11,72518,26,2615,284,5858,476,2692,8541,26,9876,284,3162,71187,1283,6849,5858,13,2308,16148,13,151645,198,151644,872,198,40,646,1487,304,11,714,18458,22191,36137,279,3766,13,151645,198,151644,77091,198],\"stop_reason\":\"eos\",\"predicted_label\":\"bug\",\"valid_label\":true,\"correct\":true},{\"wall_s\":0.0567263370001001,\"output_tokens\":2,\"output_token_ids\":[5211,151645],\"output\":\"access\",\"case_id\":\"g2\",\"expected_label\":\"bug\",\"rendered_chat\":\"<|im_start|>system\\nClassify the support ticket. Reply with exactly one lowercase label: billing, access, or bug. billing = payments, invoices, refunds; access = login or account permissions; bug = software malfunction after successful login. No explanation.<|im_end|>\\n<|im_start|>user\\nAfter successful login, the Save button does nothing and my edits disappear.<|im_end|>\\n<|im_start|>assistant\\n\",\"input_token_ids\":[151644,8948,198,1957,1437,279,1824,11727,13,17841,448,6896,825,42047,2383,25,33531,11,2615,11,476,9876,13,33531,284,14173,11,64052,11,72518,26,2615,284,5858,476,2692,8541,26,9876,284,3162,71187,1283,6849,5858,13,2308,16148,13,151645,198,151644,872,198,6025,6849,5858,11,279,10255,3137,1558,4302,323,847,50844,31053,13,151645,198,151644,77091,198],\"stop_reason\":\"eos\",\"predicted_label\":\"access\",\"valid_label\":true,\"correct\":false},{\"wall_s\":0.05651553600000625,\"output_tokens\":2,\"output_token_ids\":[2313,151645],\"output\":\"bug\",\"case_id\":\"g3\",\"expected_label\":\"bug\",\"rendered_chat\":\"<|im_start|>system\\nClassify the support ticket. Reply with exactly one lowercase label: billing, access, or bug. billing = payments, invoices, refunds; access = login or account permissions; bug = software malfunction after successful login. No explanation.<|im_end|>\\n<|im_start|>user\\nI am signed in. The search page shows an error for every search query.<|im_end|>\\n<|im_start|>assistant\\n\",\"input_token_ids\":[151644,8948,198,1957,1437,279,1824,11727,13,17841,448,6896,825,42047,2383,25,33531,11,2615,11,476,9876,13,33531,284,14173,11,64052,11,72518,26,2615,284,5858,476,2692,8541,26,9876,284,3162,71187,1283,6849,5858,13,2308,16148,13,151645,198,151644,872,198,40,1079,8499,304,13,576,2711,2150,4933,458,1465,369,1449,2711,3239,13,151645,198,151644,77091,198],\"stop_reason\":\"eos\",\"predicted_label\":\"bug\",\"valid_label\":true,\"correct\":true},{\"wall_s\":0.05977478199997677,\"output_tokens\":2,\"output_token_ids\":[5211,151645],\"output\":\"access\",\"case_id\":\"g4\",\"expected_label\":\"bug\",\"rendered_chat\":\"<|im_start|>system\\nClassify the support ticket. Reply with exactly one lowercase label: billing, access, or bug. billing = payments, invoices, refunds; access = login or account permissions; bug = software malfunction after successful login. No explanation.<|im_end|>\\n<|im_start|>user\\nMy account works, but every image I upload is displayed upside down.<|im_end|>\\n<|im_start|>assistant\\n\",\"input_token_ids\":[151644,8948,198,1957,1437,279,1824,11727,13,17841,448,6896,825,42047,2383,25,33531,11,2615,11,476,9876,13,33531,284,14173,11,64052,11,72518,26,2615,284,5858,476,2692,8541,26,9876,284,3162,71187,1283,6849,5858,13,2308,16148,13,151645,198,151644,872,198,5050,2692,4278,11,714,1449,2168,358,8135,374,12596,35841,1495,13,151645,198,151644,77091,198],\"stop_reason\":\"eos\",\"predicted_label\":\"access\",\"valid_label\":true,\"correct\":false}]}")
CASES=RECORDED_TASK['cases']
TASK_ROWS=[prediction_row(s['case_id'],s['expected_label'],s['output'],finish_reason='stop' if s['stop_reason']=='eos' else 'length') for s in RECORDED_TASK['samples']]


In [ ]:
TASK_SCORES=score_predictions(TASK_ROWS)
assert (TASK_SCORES['correct'],TASK_SCORES['valid'],TASK_SCORES['attempts'])==(6,12,12)
print('Baseline and per-class recall:',task_summary(TASK_ROWS))
by_id={c['id']:c for c in CASES}
for row in TASK_ROWS: print(by_id[row['case_id']]['text'],'| gold:',row['expected'],'predicted:',row['predicted'])
example=next(s for s in RECORDED_TASK['samples'] if s['case_id']=='g2')
print(example['rendered_chat'])
print('Input positions:',len(example['input_token_ids']),'output IDs:',example['output_token_ids'])
ERROR_EXPLANATION = ""


## C2. Plot, then change one observation

Confusion matrix: true rows, predicted columns, counts over all attempts. Its fourth column retains invalid/incomplete output. Compare it with the toy next-token distribution after your sampling change.

Turn one synthetic correct label into a partial response and recompute: plausible partial text must not boost accuracy. Explain an off-diagonal cell and both axes. Optional transport plotting uses another request; it is not the main ML result.


In [ ]:
plot_matrix(TASK_ROWS,score_predictions,'Historical Transformers FP16 / calibration: 12 tickets')
plot_sampling(TOY_PROBS)
CHANGED_ROWS=[dict(r) for r in FIXTURE_ROWS]
CHANGED_ROWS[0]=prediction_row('e1','billing','billing',status='partial')
assert score_predictions(CHANGED_ROWS)['correct']==3
print(score_predictions(CHANGED_ROWS))
PLOT_INTERPRETATION = ""
def plot_transport(raw):  # Optional provided transport analysis.
    import matplotlib.pyplot as plt
    fig,axes=plt.subplots(1,2,figsize=(12,4),layout='constrained')
    kinds=['role_or_empty','content','finish','usage','done']
    for e in raw['events']: axes[0].scatter(e['at_s']*1000,kinds.index(e['kind']))
    axes[0].set(yticks=range(5),yticklabels=kinds,xlabel='Client arrival (ms)')
    axes[1].hist([len(bytes.fromhex(c['hex'])) for c in raw['chunks']],bins=[0,128,256,384,512])
    axes[1].set(xlabel='Read size (bytes)',ylabel='Number of reads')
    plt.show()


### Provided historical stream replay

Separate RTX 5070 Ti/vLLM greeting: 33 input tokens, nine content events, ten reported output tokens. Repartition bytes without changing text. No new timing is measured. Understand complete vs partial; UTF-8/SSE implementation is optional reference.


In [ ]:
#@title Optional historical stream replay — run unchanged { display-mode: "form" }
RECORDED_SERVICE=json.loads("{\n  \"contract\": \"topic-04-api-smoke-v3\",\n  \"status\": \"complete\",\n  \"model_id\": \"Qwen/Qwen2.5-0.5B-Instruct\",\n  \"revision\": \"7ae557604adf67be50417f59c2c2f167def9a775\",\n  \"served_name\": \"topic04\",\n  \"utc_started\": \"2026-10-01T22:05:29.847952+00:00\",\n  \"vllm\": \"0.29.0\",\n  \"openai_sdk\": \"3.20.0\",\n  \"httpx\": \"0.28.1\",\n  \"transformers\": \"5.17.0\",\n  \"torch\": \"2.13.0+cu130\",\n  \"cuda\": \"13.0\",\n  \"gpu\": \"NVIDIA GeForce RTX 5070 Ti\",\n  \"launch_args\": [\n    \"serve\",\n    \"<pinned-local-snapshot>\",\n    \"--served-model-name\",\n    \"topic04\",\n    \"--host\",\n    \"127.0.0.1\",\n    \"--port\",\n    \"8000\",\n    \"--dtype\",\n    \"float16\",\n    \"--max-model-len\",\n    \"2048\",\n    \"--max-num-seqs\",\n    \"4\",\n    \"--gpu-memory-utilization\",\n    \"0.45\",\n    \"--enforce-eager\"\n  ],\n  \"wsl_v1_runner_fallback\": true,\n  \"resource_samples\": [\n    {\n      \"available_ram_gib\": 68.89566421508789,\n      \"device_used_mib\": 0\n    },\n    {\n      \"available_ram_gib\": 68.76938247680664,\n      \"device_used_mib\": 0\n    }\n  ],\n  \"request_prompt\": {\n    \"messages\": [\n      {\n        \"role\": \"user\",\n        \"content\": \"Reply with one short sentence explaining what an API server does.\"\n      }\n    ],\n    \"rendered_chat\": \"<|im_start|>system\\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\\n<|im_start|>user\\nReply with one short sentence explaining what an API server does.<|im_end|>\\n<|im_start|>assistant\\n\",\n    \"input_token_ids\": [\n      151644,\n      8948,\n      198,\n      2610,\n      525,\n      1207,\n      16948,\n      11,\n      3465,\n      553,\n      54364,\n      14817,\n      13,\n      1446,\n      525,\n      264,\n      10950,\n      17847,\n      13,\n      151645,\n      198,\n      151644,\n      872,\n      198,\n      20841,\n      448,\n      825,\n      2805,\n      11652,\n      25021,\n      1128,\n      458,\n      5333,\n      3538,\n      1558,\n      13,\n      151645,\n      198,\n      151644,\n      77091,\n      198\n    ],\n    \"prompt_tokens\": 41,\n    \"input_token_pieces\": [\n      \"<|im_start|>\",\n      \"system\",\n      \"Ċ\",\n      \"You\",\n      \"Ġare\",\n      \"ĠQ\",\n      \"wen\",\n      \",\",\n      \"Ġcreated\",\n      \"Ġby\",\n      \"ĠAlibaba\",\n      \"ĠCloud\",\n      \".\",\n      \"ĠYou\",\n      \"Ġare\",\n      \"Ġa\",\n      \"Ġhelpful\",\n      \"Ġassistant\",\n      \".\",\n      \"<|im_end|>\",\n      \"Ċ\",\n      \"<|im_start|>\",\n      \"user\",\n      \"Ċ\",\n      \"Reply\",\n      \"Ġwith\",\n      \"Ġone\",\n      \"Ġshort\",\n      \"Ġsentence\",\n      \"Ġexplaining\",\n      \"Ġwhat\",\n      \"Ġan\",\n      \"ĠAPI\",\n      \"Ġserver\",\n      \"Ġdoes\",\n      \".\",\n      \"<|im_end|>\",\n      \"Ċ\",\n      \"<|im_start|>\",\n      \"assistant\",\n      \"Ċ\"\n    ]\n  },\n  \"raw_prompt\": {\n    \"messages\": [\n      {\n        \"role\": \"user\",\n        \"content\": \"Say hello briefly.\"\n      }\n    ],\n    \"rendered_chat\": \"<|im_start|>system\\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\\n<|im_start|>user\\nSay hello briefly.<|im_end|>\\n<|im_start|>assistant\\n\",\n    \"input_token_ids\": [\n      151644,\n      8948,\n      198,\n      2610,\n      525,\n      1207,\n      16948,\n      11,\n      3465,\n      553,\n      54364,\n      14817,\n      13,\n      1446,\n      525,\n      264,\n      10950,\n      17847,\n      13,\n      151645,\n      198,\n      151644,\n      872,\n      198,\n      45764,\n      23811,\n      26753,\n      13,\n      151645,\n      198,\n      151644,\n      77091,\n      198\n    ],\n    \"prompt_tokens\": 33,\n    \"input_token_pieces\": [\n      \"<|im_start|>\",\n      \"system\",\n      \"Ċ\",\n      \"You\",\n      \"Ġare\",\n      \"ĠQ\",\n      \"wen\",\n      \",\",\n      \"Ġcreated\",\n      \"Ġby\",\n      \"ĠAlibaba\",\n      \"ĠCloud\",\n      \".\",\n      \"ĠYou\",\n      \"Ġare\",\n      \"Ġa\",\n      \"Ġhelpful\",\n      \"Ġassistant\",\n      \".\",\n      \"<|im_end|>\",\n      \"Ċ\",\n      \"<|im_start|>\",\n      \"user\",\n      \"Ċ\",\n      \"Say\",\n      \"Ġhello\",\n      \"Ġbriefly\",\n      \".\",\n      \"<|im_end|>\",\n      \"Ċ\",\n      \"<|im_start|>\",\n      \"assistant\",\n      \"Ċ\"\n    ]\n  },\n  \"limits\": {\n    \"max_model_len\": 2048,\n    \"max_num_seqs\": 4,\n    \"gpu_memory_utilization\": 0.45\n  },\n  \"limitations\": [\n    \"single sequential API smoke, not throughput benchmark\",\n    \"first call may include lazy runtime initialization\",\n    \"TCP/HTTP chunk boundaries are not generated token boundaries\",\n    \"client disconnect reclamation not measured\"\n  ],\n  \"advertised_models\": [\n    \"topic04\"\n  ],\n  \"startup_s\": 24.456222492999586,\n  \"nonstream\": {\n    \"request\": {\n      \"model\": \"topic04\",\n      \"messages\": [\n        {\n          \"role\": \"user\",\n          \"content\": \"Reply with one short sentence explaining what an API server does.\"\n        }\n      ],\n      \"max_tokens\": 48,\n      \"temperature\": 0\n    },\n    \"wall_s\": 0.8830939609997586,\n    \"finish_reason\": \"stop\",\n    \"usage\": {\n      \"completion_tokens\": 29,\n      \"prompt_tokens\": 41,\n      \"total_tokens\": 70,\n      \"completion_tokens_details\": null,\n      \"prompt_tokens_details\": null\n    },\n    \"content\": \"An API server is a software application that acts as the gateway for clients to communicate with another system or service over an internet protocol (HTTP).\"\n  },\n  \"sdk_stream\": {\n    \"first_content_s\": 0.06405494099999487,\n    \"wall_s\": 0.5953651939998963,\n    \"finish_reason\": \"stop\",\n    \"content\": \"An API server is a software application that acts as the gateway for clients to communicate with another system or service over an internet protocol (HTTP).\",\n    \"sdk_chunks\": 30\n  },\n  \"raw_sse\": {\n    \"status\": \"complete\",\n    \"content\": \"Hello! How can I assist you today?\",\n    \"finish_reason\": \"stop\",\n    \"done\": true,\n    \"usage\": {\n      \"prompt_tokens\": 33,\n      \"total_tokens\": 43,\n      \"completion_tokens\": 10\n    },\n    \"first_event_s\": 0.05866083799992339,\n    \"first_content_s\": 0.05872152499978256,\n    \"wall_s\": 0.17361840400008077,\n    \"byte_count\": 2718,\n    \"event_count\": 13,\n    \"events\": [\n      {\n        \"index\": 0,\n        \"at_s\": 0.05866083799992339,\n        \"event\": \"message\",\n        \"data\": \"{\\\"id\\\":\\\"chatcmpl-ba4deba68c98bed8\\\",\\\"object\\\":\\\"chat.completion.chunk\\\",\\\"created\\\":1790892355,\\\"model\\\":\\\"topic04\\\",\\\"choices\\\":[{\\\"index\\\":0,\\\"delta\\\":{\\\"role\\\":\\\"assistant\\\",\\\"content\\\":\\\"\\\"},\\\"logprobs\\\":null,\\\"finish_reason\\\":null}],\\\"prompt_token_ids\\\":null,\\\"prompt_text\\\":null}\",\n        \"kind\": \"role_or_empty\"\n      },\n      {\n        \"index\": 1,\n        \"at_s\": 0.05872152499978256,\n        \"event\": \"message\",\n        \"data\": \"{\\\"id\\\":\\\"chatcmpl-ba4deba68c98bed8\\\",\\\"object\\\":\\\"chat.completion.chunk\\\",\\\"created\\\":1790892355,\\\"model\\\":\\\"topic04\\\",\\\"choices\\\":[{\\\"index\\\":0,\\\"delta\\\":{\\\"content\\\":\\\"Hello\\\"},\\\"logprobs\\\":null,\\\"finish_reason\\\":null,\\\"token_ids\\\":null}]}\",\n        \"kind\": \"content\"\n      },\n      {\n        \"index\": 2,\n        \"at_s\": 0.0673291999992216,\n        \"event\": \"message\",\n        \"data\": \"{\\\"id\\\":\\\"chatcmpl-ba4deba68c98bed8\\\",\\\"object\\\":\\\"chat.completion.chunk\\\",\\\"created\\\":1790892355,\\\"model\\\":\\\"topic04\\\",\\\"choices\\\":[{\\\"index\\\":0,\\\"delta\\\":{\\\"content\\\":\\\"!\\\"},\\\"logprobs\\\":null,\\\"finish_reason\\\":null,\\\"token_ids\\\":null}]}\",\n        \"kind\": \"content\"\n      },\n      {\n        \"index\": 3,\n        \"at_s\": 0.07651217199963867,\n        \"event\": \"message\",\n        \"data\": \"{\\\"id\\\":\\\"chatcmpl-ba4deba68c98bed8\\\",\\\"object\\\":\\\"chat.completion.chunk\\\",\\\"created\\\":1790892355,\\\"model\\\":\\\"topic04\\\",\\\"choices\\\":[{\\\"index\\\":0,\\\"delta\\\":{\\\"content\\\":\\\" How\\\"},\\\"logprobs\\\":null,\\\"finish_reason\\\":null,\\\"token_ids\\\":null}]}\",\n        \"kind\": \"content\"\n      },\n      {\n        \"index\": 4,\n        \"at_s\": 0.08649925200006692,\n        \"event\": \"message\",\n        \"data\": \"{\\\"id\\\":\\\"chatcmpl-ba4deba68c98bed8\\\",\\\"object\\\":\\\"chat.completion.chunk\\\",\\\"created\\\":1790892355,\\\"model\\\":\\\"topic04\\\",\\\"choices\\\":[{\\\"index\\\":0,\\\"delta\\\":{\\\"content\\\":\\\" can\\\"},\\\"logprobs\\\":null,\\\"finish_reason\\\":null,\\\"token_ids\\\":null}]}\",\n        \"kind\": \"content\"\n      },\n      {\n        \"index\": 5,\n        \"at_s\": 0.09595841199916322,\n        \"event\": \"message\",\n        \"data\": \"{\\\"id\\\":\\\"chatcmpl-ba4deba68c98bed8\\\",\\\"object\\\":\\\"chat.completion.chunk\\\",\\\"created\\\":1790892355,\\\"model\\\":\\\"topic04\\\",\\\"choices\\\":[{\\\"index\\\":0,\\\"delta\\\":{\\\"content\\\":\\\" I\\\"},\\\"logprobs\\\":null,\\\"finish_reason\\\":null,\\\"token_ids\\\":null}]}\",\n        \"kind\": \"content\"\n      },\n      {\n        \"index\": 6,\n        \"at_s\": 0.10532649800006766,\n        \"event\": \"message\",\n        \"data\": \"{\\\"id\\\":\\\"chatcmpl-ba4deba68c98bed8\\\",\\\"object\\\":\\\"chat.completion.chunk\\\",\\\"created\\\":1790892355,\\\"model\\\":\\\"topic04\\\",\\\"choices\\\":[{\\\"index\\\":0,\\\"delta\\\":{\\\"content\\\":\\\" assist\\\"},\\\"logprobs\\\":null,\\\"finish_reason\\\":null,\\\"token_ids\\\":null}]}\",\n        \"kind\": \"content\"\n      },\n      {\n        \"index\": 7,\n        \"at_s\": 0.11451939499966102,\n        \"event\": \"message\",\n        \"data\": \"{\\\"id\\\":\\\"chatcmpl-ba4deba68c98bed8\\\",\\\"object\\\":\\\"chat.completion.chunk\\\",\\\"created\\\":1790892355,\\\"model\\\":\\\"topic04\\\",\\\"choices\\\":[{\\\"index\\\":0,\\\"delta\\\":{\\\"content\\\":\\\" you\\\"},\\\"logprobs\\\":null,\\\"finish_reason\\\":null,\\\"token_ids\\\":null}]}\",\n        \"kind\": \"content\"\n      },\n      {\n        \"index\": 8,\n        \"at_s\": 0.12420863999977882,\n        \"event\": \"message\",\n        \"data\": \"{\\\"id\\\":\\\"chatcmpl-ba4deba68c98bed8\\\",\\\"object\\\":\\\"chat.completion.chunk\\\",\\\"created\\\":1790892355,\\\"model\\\":\\\"topic04\\\",\\\"choices\\\":[{\\\"index\\\":0,\\\"delta\\\":{\\\"content\\\":\\\" today\\\"},\\\"logprobs\\\":null,\\\"finish_reason\\\":null,\\\"token_ids\\\":null}]}\",\n        \"kind\": \"content\"\n      },\n      {\n        \"index\": 9,\n        \"at_s\": 0.14300291300060053,\n        \"event\": \"message\",\n        \"data\": \"{\\\"id\\\":\\\"chatcmpl-ba4deba68c98bed8\\\",\\\"object\\\":\\\"chat.completion.chunk\\\",\\\"created\\\":1790892355,\\\"model\\\":\\\"topic04\\\",\\\"choices\\\":[{\\\"index\\\":0,\\\"delta\\\":{\\\"content\\\":\\\"?\\\"},\\\"logprobs\\\":null,\\\"finish_reason\\\":null,\\\"token_ids\\\":null}]}\",\n        \"kind\": \"content\"\n      },\n      {\n        \"index\": 10,\n        \"at_s\": 0.17354318899924692,\n        \"event\": \"message\",\n        \"data\": \"{\\\"id\\\":\\\"chatcmpl-ba4deba68c98bed8\\\",\\\"object\\\":\\\"chat.completion.chunk\\\",\\\"created\\\":1790892355,\\\"model\\\":\\\"topic04\\\",\\\"choices\\\":[{\\\"index\\\":0,\\\"delta\\\":{\\\"content\\\":\\\"\\\"},\\\"logprobs\\\":null,\\\"finish_reason\\\":\\\"stop\\\",\\\"stop_reason\\\":null,\\\"token_ids\\\":null}]}\",\n        \"kind\": \"finish\"\n      },\n      {\n        \"index\": 11,\n        \"at_s\": 0.17358486999910383,\n        \"event\": \"message\",\n        \"data\": \"{\\\"id\\\":\\\"chatcmpl-ba4deba68c98bed8\\\",\\\"object\\\":\\\"chat.completion.chunk\\\",\\\"created\\\":1790892355,\\\"model\\\":\\\"topic04\\\",\\\"choices\\\":[],\\\"usage\\\":{\\\"prompt_tokens\\\":33,\\\"total_tokens\\\":43,\\\"completion_tokens\\\":10},\\\"system_fingerprint\\\":\\\"vllm-0.29.0-nohash\\\"}\",\n        \"kind\": \"usage\"\n      },\n      {\n        \"index\": 12,\n        \"at_s\": 0.17361840400008077,\n        \"event\": \"message\",\n        \"data\": \"[DONE]\",\n        \"kind\": \"done\"\n      }\n    ],\n    \"error\": null,\n    \"unfinished_frame\": false,\n    \"clock\": \"client chunk-arrival wall; not token emission or GPU time\",\n    \"request\": {\n      \"model\": \"topic04\",\n      \"messages\": [\n        {\n          \"role\": \"user\",\n          \"content\": \"Say hello briefly.\"\n        }\n      ],\n      \"max_tokens\": 16,\n      \"temperature\": 0,\n      \"stream\": true,\n      \"stream_options\": {\n        \"include_usage\": true\n      }\n    },\n    \"http_status\": 200,\n    \"content_type\": \"text/event-stream\",\n    \"chunks\": [\n      {\n        \"at_s\": 0.05866083799992339,\n        \"hex\": \"646174613a207b226964223a2263686174636d706c2d62613464656261363863393862656438222c226f626a656374223a22636861742e636f6d706c6574696f6e2e6368756e6b222c2263726561746564223a313739303839323335352c226d6f64656c223a22746f7069633034222c2263686f69636573223a5b7b22696e646578223a302c2264656c7461223a7b22726f6c65223a22617373697374616e74222c22636f6e74656e74223a22227d2c226c6f6770726f6273223a6e756c6c2c2266696e6973685f726561736f6e223a6e756c6c7d5d2c2270726f6d70745f746f6b656e5f696473223a6e756c6c2c2270726f6d70745f74657874223a6e756c6c7d0a0a\"\n      },\n      {\n        \"at_s\": 0.05872152499978256,\n        \"hex\": \"646174613a207b226964223a2263686174636d706c2d62613464656261363863393862656438222c226f626a656374223a22636861742e636f6d706c6574696f6e2e6368756e6b222c2263726561746564223a313739303839323335352c226d6f64656c223a22746f7069633034222c2263686f69636573223a5b7b22696e646578223a302c2264656c7461223a7b22636f6e74656e74223a2248656c6c6f227d2c226c6f6770726f6273223a6e756c6c2c2266696e6973685f726561736f6e223a6e756c6c2c22746f6b656e5f696473223a6e756c6c7d5d7d0a0a\"\n      },\n      {\n        \"at_s\": 0.0673291999992216,\n        \"hex\": \"646174613a207b226964223a2263686174636d706c2d62613464656261363863393862656438222c226f626a656374223a22636861742e636f6d706c6574696f6e2e6368756e6b222c2263726561746564223a313739303839323335352c226d6f64656c223a22746f7069633034222c2263686f69636573223a5b7b22696e646578223a302c2264656c7461223a7b22636f6e74656e74223a2221227d2c226c6f6770726f6273223a6e756c6c2c2266696e6973685f726561736f6e223a6e756c6c2c22746f6b656e5f696473223a6e756c6c7d5d7d0a0a\"\n      },\n      {\n        \"at_s\": 0.07651217199963867,\n        \"hex\": \"646174613a207b226964223a2263686174636d706c2d62613464656261363863393862656438222c226f626a656374223a22636861742e636f6d706c6574696f6e2e6368756e6b222c2263726561746564223a313739303839323335352c226d6f64656c223a22746f7069633034222c2263686f69636573223a5b7b22696e646578223a302c2264656c7461223a7b22636f6e74656e74223a2220486f77227d2c226c6f6770726f6273223a6e756c6c2c2266696e6973685f726561736f6e223a6e756c6c2c22746f6b656e5f696473223a6e756c6c7d5d7d0a0a\"\n      },\n      {\n        \"at_s\": 0.08649925200006692,\n        \"hex\": \"646174613a207b226964223a2263686174636d706c2d62613464656261363863393862656438222c226f626a656374223a22636861742e636f6d706c6574696f6e2e6368756e6b222c2263726561746564223a313739303839323335352c226d6f64656c223a22746f7069633034222c2263686f69636573223a5b7b22696e646578223a302c2264656c7461223a7b22636f6e74656e74223a222063616e227d2c226c6f6770726f6273223a6e756c6c2c2266696e6973685f726561736f6e223a6e756c6c2c22746f6b656e5f696473223a6e756c6c7d5d7d0a0a\"\n      },\n      {\n        \"at_s\": 0.09595841199916322,\n        \"hex\": \"646174613a207b226964223a2263686174636d706c2d62613464656261363863393862656438222c226f626a656374223a22636861742e636f6d706c6574696f6e2e6368756e6b222c2263726561746564223a313739303839323335352c226d6f64656c223a22746f7069633034222c2263686f69636573223a5b7b22696e646578223a302c2264656c7461223a7b22636f6e74656e74223a222049227d2c226c6f6770726f6273223a6e756c6c2c2266696e6973685f726561736f6e223a6e756c6c2c22746f6b656e5f696473223a6e756c6c7d5d7d0a0a\"\n      },\n      {\n        \"at_s\": 0.10532649800006766,\n        \"hex\": \"646174613a207b226964223a2263686174636d706c2d62613464656261363863393862656438222c226f626a656374223a22636861742e636f6d706c6574696f6e2e6368756e6b222c2263726561746564223a313739303839323335352c226d6f64656c223a22746f7069633034222c2263686f69636573223a5b7b22696e646578223a302c2264656c7461223a7b22636f6e74656e74223a2220617373697374227d2c226c6f6770726f6273223a6e756c6c2c2266696e6973685f726561736f6e223a6e756c6c2c22746f6b656e5f696473223a6e756c6c7d5d7d0a0a\"\n      },\n      {\n        \"at_s\": 0.11451939499966102,\n        \"hex\": \"646174613a207b226964223a2263686174636d706c2d62613464656261363863393862656438222c226f626a656374223a22636861742e636f6d706c6574696f6e2e6368756e6b222c2263726561746564223a313739303839323335352c226d6f64656c223a22746f7069633034222c2263686f69636573223a5b7b22696e646578223a302c2264656c7461223a7b22636f6e74656e74223a2220796f75227d2c226c6f6770726f6273223a6e756c6c2c2266696e6973685f726561736f6e223a6e756c6c2c22746f6b656e5f696473223a6e756c6c7d5d7d0a0a\"\n      },\n      {\n        \"at_s\": 0.12420863999977882,\n        \"hex\": \"646174613a207b226964223a2263686174636d706c2d62613464656261363863393862656438222c226f626a656374223a22636861742e636f6d706c6574696f6e2e6368756e6b222c2263726561746564223a313739303839323335352c226d6f64656c223a22746f7069633034222c2263686f69636573223a5b7b22696e646578223a302c2264656c7461223a7b22636f6e74656e74223a2220746f646179227d2c226c6f6770726f6273223a6e756c6c2c2266696e6973685f726561736f6e223a6e756c6c2c22746f6b656e5f696473223a6e756c6c7d5d7d0a0a\"\n      },\n      {\n        \"at_s\": 0.14300291300060053,\n        \"hex\": \"646174613a207b226964223a2263686174636d706c2d62613464656261363863393862656438222c226f626a656374223a22636861742e636f6d706c6574696f6e2e6368756e6b222c2263726561746564223a313739303839323335352c226d6f64656c223a22746f7069633034222c2263686f69636573223a5b7b22696e646578223a302c2264656c7461223a7b22636f6e74656e74223a223f227d2c226c6f6770726f6273223a6e756c6c2c2266696e6973685f726561736f6e223a6e756c6c2c22746f6b656e5f696473223a6e756c6c7d5d7d0a0a\"\n      },\n      {\n        \"at_s\": 0.17354318899924692,\n        \"hex\": \"646174613a207b226964223a2263686174636d706c2d62613464656261363863393862656438222c226f626a656374223a22636861742e636f6d706c6574696f6e2e6368756e6b222c2263726561746564223a313739303839323335352c226d6f64656c223a22746f7069633034222c2263686f69636573223a5b7b22696e646578223a302c2264656c7461223a7b22636f6e74656e74223a22227d2c226c6f6770726f6273223a6e756c6c2c2266696e6973685f726561736f6e223a2273746f70222c2273746f705f726561736f6e223a6e756c6c2c22746f6b656e5f696473223a6e756c6c7d5d7d0a0a\"\n      },\n      {\n        \"at_s\": 0.17358486999910383,\n        \"hex\": \"646174613a207b226964223a2263686174636d706c2d62613464656261363863393862656438222c226f626a656374223a22636861742e636f6d706c6574696f6e2e6368756e6b222c2263726561746564223a313739303839323335352c226d6f64656c223a22746f7069633034222c2263686f69636573223a5b5d2c227573616765223a7b2270726f6d70745f746f6b656e73223a33332c22746f74616c5f746f6b656e73223a34332c22636f6d706c6574696f6e5f746f6b656e73223a31307d2c2273797374656d5f66696e6765727072696e74223a22766c6c6d2d302e32392e302d6e6f68617368227d0a0a\"\n      },\n      {\n        \"at_s\": 0.17361840400008077,\n        \"hex\": \"646174613a205b444f4e455d0a0a\"\n      }\n    ]\n  },\n  \"negative\": {\n    \"wrong_model\": {\n      \"status\": 404\n    },\n    \"over_context\": {\n      \"status\": 400,\n      \"rendered_prompt_tokens\": 2329\n    }\n  },\n  \"teardown\": {\n    \"owned_process_group_stopped\": true,\n    \"server_returncode\": 0\n  },\n  \"utc_finished\": \"2026-10-01T22:05:56.753374+00:00\"\n}\n")
raw=RECORDED_SERVICE['raw_sse']
REAL_WIRE=b''.join(bytes.fromhex(c['hex']) for c in raw['chunks'])
REPLAY=assemble_response(decode_chunks([bytes.fromhex(c['hex']) for c in raw['chunks']]))
assert REPLAY['content']==raw['content'] and REPLAY['status']=='complete'
PARTITION='seventeen_bytes'
changed=byte_partitions(REAL_WIRE)[PARTITION]
assert assemble_response(decode_chunks(changed))==REPLAY
# plot_transport(raw)  # Optional.


## C3. Declare the comparison, not its winner

Choose CHALLENGER='prompt' or 'format'. Prompt changes only the system instruction to distinguish inability to log in from broken features after login. Format changes only structured_outputs.choice; the prompt stays identical. Model/revision, cases, temperature zero and cap are matched. Read both literal instructions before running.

Pinned Qwen's isolated labels are single IDs: billing=[38637], access=[5211], bug=[2313]. This does not guarantee the first generated token is a label. D prints actual tokenizer IDs. A cap-one failure needs its actual output/finish reason; do not call it bill+ing truncation without evidence.

Default cap 8 is shared by both candidates. Before measuring, write which change you expect and what unchanged result would mean. Historical 100% validity suggests a format constraint may add no useful benefit. Select a winner only after observing D; public held-out teaching data is opened in E, not in this plan.


In [ ]:
OUTPUT_CAP=8
CHALLENGER='prompt'  # Change to format for a syntax-control experiment.
CANDIDATES=task_namespace['comparison_policies'](CHALLENGER,OUTPUT_CAP)
for policy in CANDIDATES: print(policy)
from pathlib import Path
CPU_DECISION=dict(contract='topic04-client-investigation-v3',scope='CPU plan; not a chosen live policy',admission_checks=ADMISSION_CHECKS,
    scoring_checks=SCORING_CHECKS,historical_task_scores=TASK_SCORES,
    candidates=CANDIDATES,
    identity_prediction=IDENTITY_PREDICTION,sampling_prediction=SAMPLING_PREDICTION,
    error_explanation=ERROR_EXPLANATION,plot_interpretation=PLOT_INTERPRETATION)
assert all(CPU_DECISION[k].strip() for k in ('identity_prediction','sampling_prediction','error_explanation','plot_interpretation'))
Path('evidence').mkdir(exist_ok=True)
Path('evidence/topic-04-client.json').write_text(json.dumps(CPU_DECISION,indent=2,ensure_ascii=False))
print('CPU plan saved; winner not selected; no GPU evidence')


## D1. Safe runtime, supplied infrastructure

Keep RUN_GPU=False until A–C pass and the instructor confirms resources. The read-only gate runs **before installation**: 24 GiB available RAM, GPU used memory at most 12 GiB, no other compute process. It is a course safety gate, not an OS sandbox or validated T4 minimum. Never lower it to force a run.

The runner owns one process group, refuses an occupied port, binds loopback and reuses a pinned cache. No tunnel or production keys. If installation needs a kernel restart, restart and repeat CPU/preflight. Stop on pressure or cleanup failure.


In [ ]:
#@title Provided GPU runner — opt in only after preflight { display-mode: "form" }
lab_namespace={'__name__':'topic04_student_lab'}
exec("\"\"\"Bounded UTF-8/SSE decoder + single-choice chat lifecycle, standard library only.\n\nFraming follows WHATWG event streams (CR/LF/CRLF, comments, multiline data).\nChat completion is a separate, deliberately strict course contract, not all SSE.\n\"\"\"\nimport codecs\nimport json\nimport math\n\n\nclass SSEProtocolError(ValueError):\n    pass\n\n\nclass SSEDecoder:\n    def __init__(self, max_bytes=262144, max_events=4096):\n        self.decoder = codecs.getincrementaldecoder('utf-8-sig')('strict')\n        self.line = ''; self.data = []; self.event = ''; self.after_cr = False\n        self.bytes = 0; self.count = 0; self.max_bytes = max_bytes; self.max_events = max_events\n        self.closed = False\n\n    def _line(self):\n        line, self.line = self.line, ''\n        if not line:\n            if not self.data:\n                self.event = ''; return None\n            event = dict(event=self.event or 'message', data='\\n'.join(self.data))\n            self.data = []; self.event = ''; self.count += 1\n            if self.count > self.max_events: raise SSEProtocolError('event limit exceeded')\n            return event\n        if line.startswith(':'): return None\n        field, sep, value = line.partition(':')\n        if value.startswith(' '): value = value[1:]\n        if field == 'data': self.data.append(value)\n        elif field == 'event': self.event = value\n        # id/retry/unknown fields do not alter this finite HTTP replay client.\n        return None\n\n    def feed(self, raw):\n        if self.closed: raise SSEProtocolError('decoder already closed')\n        if not isinstance(raw,bytes): raise TypeError('feed requires bytes, not decoded strings')\n        self.bytes += len(raw)\n        if self.bytes > self.max_bytes: raise SSEProtocolError('byte limit exceeded')\n        text = self.decoder.decode(raw)\n        events = []\n        for char in text:\n            if self.after_cr:\n                self.after_cr = False\n                if char == '\\n': continue\n            if char in '\\r\\n':\n                event = self._line()\n                if event is not None: events.append(event)\n                self.after_cr = char == '\\r'\n            else: self.line += char\n        return events\n\n    def finish(self):\n        self.decoder.decode(b'',final=True)  # Incomplete UTF-8 is a protocol error.\n        self.closed = True\n        # WHATWG does not dispatch an unfinished event at EOF.\n        return bool(self.line or self.data)\n\n\nclass ChatStream:\n    \"\"\"No reconnect/retry: partial text is evidence, never a completed answer.\"\"\"\n    def __init__(self):\n        self.decoder = SSEDecoder(); self.events = []; self.content = ''\n        self.finish_reason = None; self.usage = None; self.done = False\n        self.first_event_s = None; self.first_content_s = None; self.last_s = 0.\n        self.error = None; self.closed = False; self.cancelled = False; self.unfinished_frame = False\n\n    def _event(self,event,at_s):\n        if self.done: raise SSEProtocolError('data after DONE')\n        if event['event'] != 'message': raise SSEProtocolError('only message events supported')\n        if self.first_event_s is None: self.first_event_s = at_s\n        body=event['data']\n        row=dict(index=len(self.events),at_s=at_s,event=event['event'],data=body)\n        self.events.append(row)\n        if body == '[DONE]':\n            if self.finish_reason is None: raise SSEProtocolError('DONE without finish_reason')\n            self.done = True; row['kind']='done'; return\n        obj=json.loads(body)\n        if not isinstance(obj,dict): raise SSEProtocolError('JSON object required')\n        if 'error' in obj: raise SSEProtocolError('server error event')\n        choices=obj.get('choices')\n        if not isinstance(choices,list): raise SSEProtocolError('choices list required')\n        usage=obj.get('usage')\n        if usage is not None:\n            if not isinstance(usage,dict): raise SSEProtocolError('invalid usage')\n            for key in ('prompt_tokens','completion_tokens','total_tokens'):\n                if type(usage.get(key)) is not int or usage[key]<0: raise SSEProtocolError('invalid usage counts')\n            if usage['total_tokens'] != usage['prompt_tokens']+usage['completion_tokens']:\n                raise SSEProtocolError('usage total mismatch')\n            self.usage=usage\n        if not choices:\n            if usage is None: raise SSEProtocolError('empty choices without usage')\n            row['kind']='usage'; return\n        if len(choices)!=1 or not isinstance(choices[0],dict) or type(choices[0].get('index')) is not int or choices[0]['index']!=0:\n            raise SSEProtocolError('only object choice zero supported')\n        choice=choices[0]; delta=choice.get('delta',{})\n        if not isinstance(delta,dict): raise SSEProtocolError('delta object required')\n        if any(key not in ('role','content') and value is not None for key,value in delta.items()):\n            raise SSEProtocolError('unsupported non-text delta')\n        if delta.get('role') not in (None,'assistant'): raise SSEProtocolError('unsupported role')\n        text=delta.get('content')\n        if text is not None and not isinstance(text,str): raise SSEProtocolError('content must be text or null')\n        if text:\n            if self.finish_reason is not None: raise SSEProtocolError('content after finish')\n            if self.first_content_s is None: self.first_content_s=at_s\n            self.content += text\n        finish=choice.get('finish_reason')\n        if finish is not None:\n            if finish not in ('stop','length') or self.finish_reason is not None:\n                raise SSEProtocolError('invalid or duplicate finish')\n            self.finish_reason=finish\n        row['kind']='finish' if finish is not None else ('content' if text else 'role_or_empty')\n\n    def feed(self,raw,at_s):\n        try:\n            if self.closed: raise SSEProtocolError('stream already closed')\n            if self.error: raise SSEProtocolError('stream already failed')\n            if type(at_s) not in (int,float) or not math.isfinite(at_s) or at_s<self.last_s:\n                raise SSEProtocolError('invalid or nonmonotonic arrival clock')\n            self.last_s=at_s\n            for event in self.decoder.feed(raw): self._event(event,at_s)\n        except (ValueError,TypeError,UnicodeError) as exc:\n            self.error=type(exc).__name__ + ': ' + str(exc)\n            raise\n\n    def finish(self,cancelled=False,transport_error=None):\n        if self.closed: raise SSEProtocolError('stream already closed')\n        self.closed=True; self.cancelled=cancelled\n        try: self.unfinished_frame=self.decoder.finish()\n        except UnicodeError as exc: self.error=type(exc).__name__ + ': incomplete UTF-8'\n        if transport_error: self.error=str(transport_error)\n        if self.error: status='failed'\n        elif cancelled: status='cancelled'\n        elif self.done and self.finish_reason is not None and not self.unfinished_frame: status='complete'\n        else: status='partial'\n        return dict(status=status,content=self.content,finish_reason=self.finish_reason,done=self.done,\n                    usage=self.usage,first_event_s=self.first_event_s,first_content_s=self.first_content_s,\n                    wall_s=self.last_s,byte_count=self.decoder.bytes,event_count=len(self.events),\n                    events=self.events,error=self.error,unfinished_frame=self.unfinished_frame,\n                    clock='client chunk-arrival wall; not token emission or GPU time')\n\n\ndef synthetic_fixture():\n    def chunk(delta,finish=None):\n        return dict(id='illustrative',object='chat.completion.chunk',model='topic04',\n                    choices=[dict(index=0,delta=delta,finish_reason=finish)])\n    payload=dict(model='topic04',messages=[dict(role='user',content='Say café ☕ briefly.')],\n                 max_tokens=16,temperature=0,stream=True,stream_options=dict(include_usage=True))\n    bodies=[chunk(dict(role='assistant',content='')),chunk(dict(content='Café ')),\n            chunk(dict(content='☕.')),chunk({},'stop'),\n            dict(choices=[],usage=dict(prompt_tokens=13,completion_tokens=4,total_tokens=17))]\n    # The synthetic usage counts are invented, not tokenizer measurements.\n    frames=[': heartbeat\\r\\n\\r\\n']+['data: '+json.dumps(b,ensure_ascii=False,separators=(',',':'))+'\\r\\n\\r\\n' for b in bodies]+['data: [DONE]\\r\\n\\r\\n']\n    arrival=[.01,.03,.10,.14,.15,.16,.17]\n    chunks=[]\n    for text,at_s in zip(frames,arrival):\n        raw=text.encode('utf-8')\n        # Always split CRLF; split UTF-8 continuation sequences when present.\n        cuts=sorted(set([len(raw)-3]+[i+1 for i,b in enumerate(raw) if b in (0xc3,0xe2)]))\n        previous=0\n        for end in cuts+[len(raw)]:\n            chunks.append(dict(at_s=at_s,hex=raw[previous:end].hex())); previous=end\n    stream=ChatStream()\n    for c in chunks: stream.feed(bytes.fromhex(c['hex']),c['at_s'])\n    complete=stream.finish()\n    prefix=ChatStream()\n    for c in chunks:\n        if c['at_s']>.10: break\n        prefix.feed(bytes.fromhex(c['hex']),c['at_s'])\n    partial=prefix.finish()\n    cancelled=ChatStream()\n    for c in chunks:\n        if c['at_s']>.10: break\n        cancelled.feed(bytes.fromhex(c['hex']),c['at_s'])\n    return dict(schema='topic04-sse-fixture-v1',kind='illustrative bytes, clocks and usage; no server or tokenizer executed',\n                request=payload,frames=frames,chunks=chunks,complete=complete,partial=partial,\n                cancelled=cancelled.finish(cancelled=True),source='https://html.spec.whatwg.org/multipage/server-sent-events.html')\n\n\"\"\"Canonical bounded paired smoke. Import is CPU-only; GPU work starts at run_lab().\"\"\"\nimport gc\nimport json\nimport math\nimport os\nimport statistics\nimport subprocess\nimport time\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nMODEL_ID='Qwen/Qwen2.5-0.5B-Instruct'\nREVISION='7ae557604adf67be50417f59c2c2f167def9a775'\nPROTOCOL='topic-03-paired-smoke-v3'\nTRIALS, MAX_NEW=3,24\nPROMPTS=[\n    dict(bucket='format',text='Reply with exactly one word: blue or orange. What color is the sky on a clear day?'),\n    dict(bucket='arithmetic',text='Compute 17 + 28. Give only the integer.'),\n    dict(bucket='instruction',text='Write exactly three comma-separated verbs for measuring a service.'),\n    dict(bucket='reasoning',text='A server has 2 GiB free and a new request needs 3 GiB. In one sentence, say whether it fits and why.'),\n]\n\n\ndef checkpoint(path,data):\n    path=Path(path); temporary=path.with_suffix(path.suffix+'.tmp')\n    temporary.write_text(json.dumps(data,indent=2,ensure_ascii=False,allow_nan=False)+'\\n',encoding='utf-8')\n    temporary.replace(path)\n\n\ndef resource_gate(require_idle=False):\n    mem=Path('/proc/meminfo')\n    if not mem.exists(): raise RuntimeError('Linux/WSL RAM gate unavailable; use supported CUDA runtime')\n    available=int(next(line.split()[1] for line in mem.read_text().splitlines() if line.startswith('MemAvailable:')))*1024\n    used=[int(x) for x in subprocess.check_output(['nvidia-smi','--query-gpu=memory.used','--format=csv,noheader,nounits'],text=True,timeout=10).split()]\n    if available < 24*2**30 or not used or max(used)>12288: raise RuntimeError('STOP: RAM/GPU resource gate')\n    if require_idle:\n        active=subprocess.check_output(['nvidia-smi','--query-compute-apps=pid','--format=csv,noheader,nounits'],text=True,timeout=10).strip()\n        if active: raise RuntimeError('STOP: GPU compute process already active')\n    return dict(available_ram_gib=available/2**30,device_used_mib=max(used))\n\n\ndef resolve_snapshot():\n    from huggingface_hub import snapshot_download\n    cache=Path(os.environ.get('LSM_MODEL_CACHE',Path.cwd()/'.cache/models')).expanduser()\n    try: snapshot=snapshot_download(MODEL_ID,revision=REVISION,cache_dir=str(cache),local_files_only=True)\n    except FileNotFoundError:\n        if os.environ.get('HF_HUB_OFFLINE')=='1': raise RuntimeError('Pinned offline snapshot missing; do not redownload')\n        snapshot=snapshot_download(MODEL_ID,revision=REVISION,cache_dir=str(cache))\n    return Path(snapshot)\n\n\n\n\"\"\"Bounded local API contract rehearsal. No GPU/import/network work at module import.\"\"\"\nimport atexit\nimport json\nimport os\nimport platform\nimport signal\nimport socket\nimport subprocess\nimport sys\nimport time\nimport threading\nfrom contextlib import contextmanager\nfrom datetime import datetime,timezone\nfrom pathlib import Path\n\nPROTOCOL='topic-04-api-smoke-v3'\nSERVED_NAME='topic04'\nPROMPT='Reply with one short sentence explaining what an API server does.'\nRAW_PROMPT='Say hello briefly.'\n\n\n@contextmanager\ndef request_deadline(seconds=120):\n    \"\"\"POSIX/main-thread absolute wall deadline, distinct from HTTP read inactivity.\n\n    Refuse an existing real-time alarm rather than overwrite another owner's timer.\n    Always restore the previous handler. The course runtime is Linux/WSL/Colab.\n    \"\"\"\n    if threading.current_thread() is not threading.main_thread() or not hasattr(signal, 'setitimer'):\n        raise RuntimeError('Absolute request deadline requires POSIX main thread')\n    if signal.getitimer(signal.ITIMER_REAL)[0]:\n        raise RuntimeError('An existing alarm owns this process; use an isolated runtime')\n    previous = signal.getsignal(signal.SIGALRM)\n    def expire(signum, frame):\n        raise TimeoutError('Absolute request deadline exceeded')\n    signal.signal(signal.SIGALRM, expire)\n    signal.setitimer(signal.ITIMER_REAL, seconds)\n    try:\n        yield\n    finally:\n        signal.setitimer(signal.ITIMER_REAL, 0)\n        signal.signal(signal.SIGALRM, previous)\n\n\ndef normalize_input_ids(encoded):\n    \"\"\"Transformers 4/5 may return IDs or a BatchEncoding; accept one sequence only.\"\"\"\n    value=encoded['input_ids'] if hasattr(encoded,'keys') else encoded\n    if isinstance(value,list) and len(value)==1 and isinstance(value[0],list): value=value[0]\n    if not isinstance(value,list) or not value or any(type(x) is not int or x<0 for x in value):\n        raise ValueError('Expected one nonempty integer input_ids sequence')\n    return value\n\n\ndef stop_owned_process_group(server,term_seconds=20,kill_seconds=10):\n    \"\"\"Stop only a POSIX session we created, including worker descendants.\"\"\"\n    try: os.killpg(server.pid,signal.SIGTERM)\n    except ProcessLookupError: pass\n    deadline=time.monotonic()+term_seconds\n    while time.monotonic()<deadline:\n        server.poll()\n        try: os.killpg(server.pid,0)\n        except ProcessLookupError: break\n        time.sleep(.1)\n    else:\n        try: os.killpg(server.pid,signal.SIGKILL)\n        except ProcessLookupError: pass\n    server.wait(timeout=kill_seconds)\n    deadline=time.monotonic()+kill_seconds\n    while time.monotonic()<deadline:\n        try: os.killpg(server.pid,0)\n        except ProcessLookupError: return\n        time.sleep(.1)\n    raise RuntimeError('Owned process group remains after teardown')\n\n\ndef run_lab(snapshot,output_path,task_probe=None):\n    import httpx,openai,torch,vllm,transformers\n    from transformers import AutoTokenizer\n    if vllm.__version__!='0.29.0': raise RuntimeError('Pinned vLLM 0.29.0 required')\n    output_path=Path(output_path)\n    if output_path.exists(): raise FileExistsError('Preserve prior evidence: choose a fresh output path')\n    output_path.parent.mkdir(parents=True,exist_ok=True)\n    preflight=resource_gate(require_idle=True)\n    tokenizer=AutoTokenizer.from_pretrained(str(snapshot),local_files_only=True)\n    def prompt_record(text):\n        messages=[dict(role='user',content=text)]\n        ids=normalize_input_ids(tokenizer.apply_chat_template(messages,tokenize=True,add_generation_prompt=True))\n        return dict(messages=messages,rendered_chat=tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True),\n                    input_token_ids=ids,prompt_tokens=len(ids),input_token_pieces=tokenizer.convert_ids_to_tokens(ids))\n    base='http://127.0.0.1:8000/v1'\n    with socket.socket() as probe:\n        if probe.connect_ex(('127.0.0.1',8000))==0: raise RuntimeError('Port 8000 occupied; do not stop an unowned server')\n    args=['serve',str(snapshot),'--served-model-name',SERVED_NAME,'--host','127.0.0.1','--port','8000',\n          '--dtype','float16','--max-model-len','2048','--max-num-seqs','4','--gpu-memory-utilization','0.45','--enforce-eager']\n    is_wsl=bool(os.environ.get('WSL_INTEROP')) or 'microsoft' in platform.release().lower()\n    env=os.environ.copy(); env['HF_HUB_OFFLINE']='1'; env['TRANSFORMERS_OFFLINE']='1'\n    if is_wsl: env['VLLM_USE_V2_MODEL_RUNNER']='0'; env['VLLM_USE_FLASHINFER_SAMPLER']='0'\n    data=dict(contract=PROTOCOL,status='running',model_id=MODEL_ID,revision=REVISION,served_name=SERVED_NAME,\n              utc_started=datetime.now(timezone.utc).isoformat(),vllm=vllm.__version__,openai_sdk=openai.__version__,\n              httpx=httpx.__version__,transformers=transformers.__version__,\n              torch=torch.__version__,cuda=torch.version.cuda,gpu=torch.cuda.get_device_name(0),\n              launch_args=['serve','<pinned-local-snapshot>']+args[2:],wsl_v1_runner_fallback=is_wsl,\n              resource_samples=[preflight],request_prompt=prompt_record(PROMPT),raw_prompt=prompt_record(RAW_PROMPT),\n              limits=dict(max_model_len=2048,max_num_seqs=4,gpu_memory_utilization=.45),\n              limitations=['single sequential API smoke, not throughput benchmark','first call may include lazy runtime initialization',\n                           'TCP/HTTP chunk boundaries are not generated token boundaries','client disconnect reclamation not measured'])\n    checkpoint(output_path,data)\n    # Calibration and held-out are sequential owned lifetimes in ONE attempt.\n    # Keep separate immutable logs instead of colliding at server.local.log.\n    log=output_path.with_suffix('.server.local.log').open('x',encoding='utf-8')\n    server=None; stopped=False; client=None\n    def stop_server():\n        nonlocal stopped\n        if stopped: return\n        stopped=True\n        if server is not None:\n            stop_owned_process_group(server)\n        log.close()\n    atexit.register(stop_server)\n    try:\n        server=subprocess.Popen([sys.executable,'-m','vllm.entrypoints.cli.main']+args,\n                                env=env,stdout=log,stderr=subprocess.STDOUT,start_new_session=True)\n        begin=time.monotonic(); deadline=begin+300\n        while time.monotonic()<deadline:\n            if server.poll() is not None: raise RuntimeError('vLLM exited during startup; inspect private local log')\n            resource_gate()\n            try:\n                response=httpx.get(base+'/models',timeout=2,trust_env=False)\n                if response.status_code==200:\n                    advertised=[x['id'] for x in response.json()['data']]\n                    if SERVED_NAME not in advertised: raise RuntimeError('served name mismatch')\n                    data['advertised_models']=advertised; break\n            except httpx.RequestError: pass\n            time.sleep(2)\n        else: raise TimeoutError('Startup exceeded 300 s')\n        data['startup_s']=time.monotonic()-begin; checkpoint(output_path,data)\n        client=openai.OpenAI(base_url=base,api_key='EMPTY',timeout=120,max_retries=0,\n                            http_client=httpx.Client(trust_env=False))\n        request=dict(model=SERVED_NAME,messages=[dict(role='user',content=PROMPT)],max_tokens=48,temperature=0)\n        start=time.perf_counter()\n        with request_deadline(): answer=client.chat.completions.create(**request)\n        data['nonstream']=dict(request=request,wall_s=time.perf_counter()-start,finish_reason=answer.choices[0].finish_reason,\n                               usage=answer.usage.model_dump() if answer.usage else None,content=answer.choices[0].message.content)\n        if not data['nonstream']['content']: raise RuntimeError('Empty nonstream response')\n        checkpoint(output_path,data); resource_gate()\n        start=time.perf_counter(); first=None; pieces=[]; finish=None; chunks=0\n        # Cover creation AND iteration: progressing reads must not reset wall time.\n        sdk_deadline=request_deadline(); sdk_deadline.__enter__()\n        stream=None\n        try:\n            stream=client.chat.completions.create(**request,stream=True)\n            for chunk in stream:\n                chunks+=1\n                if chunks>512: raise RuntimeError('SDK chunk budget exceeded')\n                delta=chunk.choices[0].delta.content if chunk.choices else None\n                if delta:\n                    if first is None: first=time.perf_counter()-start\n                    pieces.append(delta)\n                if chunk.choices and chunk.choices[0].finish_reason: finish=chunk.choices[0].finish_reason\n        finally:\n            try:\n                if stream is not None: stream.close()\n            finally: sdk_deadline.__exit__(None,None,None)\n        data['sdk_stream']=dict(first_content_s=first,wall_s=time.perf_counter()-start,finish_reason=finish,\n                                content=''.join(pieces),sdk_chunks=chunks)\n        if first is None or finish is None: raise RuntimeError('Incomplete SDK stream')\n        checkpoint(output_path,data); resource_gate()\n        payload=dict(model=SERVED_NAME,messages=[dict(role='user',content=RAW_PROMPT)],max_tokens=16,temperature=0,\n                     stream=True,stream_options=dict(include_usage=True))\n        parser=ChatStream(); raw_chunks=[]; start=time.perf_counter()\n        with request_deadline(), httpx.stream('POST',base+'/chat/completions',json=payload,timeout=120,trust_env=False) as response:\n            response.raise_for_status()\n            media=response.headers.get('content-type','').split(';')[0].lower()\n            if media!='text/event-stream': raise RuntimeError('Wrong SSE content type')\n            for raw in response.iter_bytes():\n                at=time.perf_counter()-start\n                raw_chunks.append(dict(at_s=at,hex=raw.hex()))\n                parser.feed(raw,at)\n        record=parser.finish(); record['request']=payload; record['http_status']=response.status_code\n        record['content_type']=media; record['chunks']=raw_chunks\n        data['raw_sse']=record; checkpoint(output_path,data)\n        if record['status']!='complete' or not record['content']: raise RuntimeError('Incomplete raw SSE')\n        negative={}\n        try:\n            with request_deadline(): client.chat.completions.create(model='not-the-served-model',messages=[dict(role='user',content='hi')],max_tokens=1)\n        except openai.APIStatusError as exc: negative['wrong_model']=dict(status=exc.status_code)\n        long_text=' blue'*2300\n        rendered=normalize_input_ids(tokenizer.apply_chat_template([dict(role='user',content=long_text)],tokenize=True,add_generation_prompt=True))\n        if not 2048<len(rendered)<8192: raise RuntimeError('Bounded over-context fixture invalid')\n        try:\n            with request_deadline(): client.chat.completions.create(model=SERVED_NAME,messages=[dict(role='user',content=long_text)],max_tokens=16)\n        except openai.APIStatusError as exc: negative['over_context']=dict(status=exc.status_code,rendered_prompt_tokens=len(rendered))\n        if set(negative)!={'wrong_model','over_context'} or negative['wrong_model']['status']!=404 or negative['over_context']['status']!=400:\n            raise RuntimeError('Expected wrong-model 404 / over-context 400 contract not met')\n        if task_probe is not None:\n            with request_deadline(): data['task_probe']=task_probe(client,tokenizer)\n        data['timeout_policy']=dict(http_phase_s=120,absolute_request_s=120,mechanism='POSIX ITIMER_REAL main thread')\n        data['negative']=negative; data['status']='complete'; checkpoint(output_path,data)\n    except BaseException as exc:\n        data['status']='failed'; data['error_type']=type(exc).__name__; raise\n    finally:\n        try:\n            if client is not None: client.close()\n            stop_server(); atexit.unregister(stop_server)\n            data['teardown']=dict(owned_process_group_stopped=True,server_returncode=None if server is None else server.returncode)\n            data['resource_samples'].append(resource_gate(require_idle=True))\n        except BaseException as exc:\n            data['status']='failed'; data['cleanup_error_type']=type(exc).__name__\n            raise\n        finally:\n            data['utc_finished']=datetime.now(timezone.utc).isoformat(); checkpoint(output_path,data)\n    return data\n\n\ndef validate_artifact(data):\n    import math\n    def require(value,message):\n        if not value: raise ValueError(message)\n    def finite_number(value): return type(value) in (int,float) and math.isfinite(value)\n    def usage_counts(usage):\n        return isinstance(usage,dict) and all(type(usage.get(k)) is int and usage[k]>=0\n            for k in ('prompt_tokens','completion_tokens','total_tokens')) and usage['completion_tokens']>0\n    require(data.get('contract')==PROTOCOL and data.get('status')=='complete','incomplete/wrong protocol')\n    require(data.get('revision')==REVISION and data.get('model_id')==MODEL_ID,'model identity')\n    require(data.get('vllm')=='0.29.0' and all(isinstance(data.get(key),str) and data[key] for key in ('openai_sdk','httpx','transformers','torch','cuda','gpu')),'runtime pins/identity')\n    require(data.get('served_name')==SERVED_NAME and SERVED_NAME in data.get('advertised_models',[]),'served identity')\n    expected_limits=dict(max_model_len=2048,max_num_seqs=4,gpu_memory_utilization=.45)\n    require(data.get('limits')==expected_limits,'fixed bounded launch limits')\n    require(type(data['limits']['max_model_len']) is int and type(data['limits']['max_num_seqs']) is int and\n            finite_number(data['limits']['gpu_memory_utilization']),'numeric launch limit types')\n    expected_launch=['serve','<pinned-local-snapshot>','--served-model-name',SERVED_NAME,'--host','127.0.0.1','--port','8000',\n                     '--dtype','float16','--max-model-len','2048','--max-num-seqs','4','--gpu-memory-utilization','0.45','--enforce-eager']\n    require(data.get('launch_args')==expected_launch,'exact loopback bounded launch')\n    require(type(data.get('wsl_v1_runner_fallback')) is bool,'explicit WSL fallback')\n    require(finite_number(data.get('startup_s')) and 0<data['startup_s']<=300,'bounded startup clock')\n    expected_request=dict(model=SERVED_NAME,messages=[dict(role='user',content=PROMPT)],max_tokens=48,temperature=0)\n    require(data['nonstream'].get('request')==expected_request,'exact nonstream request envelope')\n    require(type(data['nonstream']['request']['max_tokens']) is int and\n            finite_number(data['nonstream']['request']['temperature']),'nonstream envelope numeric types')\n    for key,text in [('request_prompt',PROMPT),('raw_prompt',RAW_PROMPT)]:\n        prompt=data[key]\n        require(prompt['messages']==[dict(role='user',content=text)],'literal request')\n        require(prompt['prompt_tokens']==len(prompt['input_token_ids'])==len(prompt['input_token_pieces'])>0,'prompt count')\n        require(all(type(x) is int and x>=0 for x in prompt['input_token_ids']),'ID types')\n    for key in ('nonstream','sdk_stream'):\n        result=data[key]\n        require(isinstance(result.get('content'),str) and bool(result['content']),'content required')\n        require(result.get('finish_reason') in ('stop','length'),'finish reason')\n        require(finite_number(result['wall_s']) and result['wall_s']>0,'wall clock')\n    require(finite_number(data['sdk_stream']['first_content_s']) and 0<data['sdk_stream']['first_content_s']<=data['sdk_stream']['wall_s'],'first content clock')\n    raw=data['raw_sse']; replay=ChatStream()\n    for chunk in raw['chunks']: replay.feed(bytes.fromhex(chunk['hex']),chunk['at_s'])\n    reconstructed=replay.finish()\n    require(all(raw[key]==value for key,value in reconstructed.items()),'raw byte replay mismatch')\n    require(raw['http_status']==200 and raw['content_type']=='text/event-stream','SSE HTTP boundary')\n    require(raw['status']=='complete' and raw['done'] and raw['content'],'partial stream is not success')\n    expected_raw=dict(model=SERVED_NAME,messages=[dict(role='user',content=RAW_PROMPT)],max_tokens=16,temperature=0,\n                      stream=True,stream_options=dict(include_usage=True))\n    require(raw.get('request')==expected_raw,'exact raw request envelope')\n    require(type(raw['request']['max_tokens']) is int and finite_number(raw['request']['temperature']) and\n            raw['request']['stream'] is True and raw['request']['stream_options']['include_usage'] is True,'raw envelope types')\n    require(raw['request']['messages']==data['raw_prompt']['messages'],'raw prompt identity')\n    require(usage_counts(raw['usage']) and raw['usage']['prompt_tokens']==data['raw_prompt']['prompt_tokens'],'raw usage vs tokenizer')\n    usage=data['nonstream']['usage']\n    require(usage_counts(usage) and usage['prompt_tokens']==data['request_prompt']['prompt_tokens'] and usage['total_tokens']==usage['prompt_tokens']+usage['completion_tokens'],'nonstream usage')\n    require(data['teardown'].get('owned_process_group_stopped') is True and type(data['teardown'].get('server_returncode')) is int,'owned server teardown')\n    resources=data.get('resource_samples',[])\n    require(len(resources)==2 and all(finite_number(r.get('available_ram_gib')) and r['available_ram_gib']>=24 and\n                type(r.get('device_used_mib')) is int and 0<=r['device_used_mib']<=12288 for r in resources),'resource envelope')\n    require(resources[-1]['device_used_mib']<=resources[0]['device_used_mib']+64,'device allocation recovered after teardown')\n    require(set(data['negative'])=={'wrong_model','over_context'},'negative probes')\n    require(all(type(row['status']) is int for row in data['negative'].values()) and\n            data['negative']['wrong_model']['status']==404 and data['negative']['over_context']['status']==400,\n            'wrong-model must be 404; over-context must be 400, not arbitrary 4xx')\n    require(type(data['negative']['over_context']['rendered_prompt_tokens']) is int and\n            2048<data['negative']['over_context']['rendered_prompt_tokens']<8192,'bounded token rejection')\n    return True\n",lab_namespace)
RUN_GPU=False #@param {type:"boolean"}
if RUN_GPU:
    assert ADMISSION_CHECKS==11 and SCORING_CHECKS==6
    print(lab_namespace['resource_gate'](require_idle=True))


In [ ]:
import subprocess,sys
from importlib.metadata import version,PackageNotFoundError
if RUN_GPU:
    lab_namespace['resource_gate'](require_idle=True)
    try: matches=version('vllm')=='0.29.0'
    except PackageNotFoundError: matches=False
    if not matches: subprocess.check_call([sys.executable,'-m','pip','install','vllm==0.29.0'])
    MODEL_DIR=lab_namespace['resolve_snapshot']()
else: print('GPU disabled: no installation/download')


## D2. Measure both candidates on calibration

Pinned Qwen FP16; context2048, max_num_seqs4, GPU memory target0.45, eager mode. Startup300s. HTTP phase/inactivity timeout120s is **not** a total limit: a POSIX main-thread alarm adds an absolute120s deadline per service request. The whole 24-ticket calibration sequence has its own120s bound. Raw stream256KiB/4096events; SDK retries0.

Both candidates run on the same 12 calibration tickets and the same server. Preserve all attempts, then plot both matrices, compare constant-access baseline and per-class recall. The server stops before you choose; held-out has not been opened. API probes check the service separately. Timeout aborts, not a passing artifact.

This is a task/integration test, not concurrency or throughput measurement. Historical Transformers vs fresh vLLM are not a controlled speed A/B.


In [ ]:
from datetime import datetime,timezone
if RUN_GPU:
    def task_probe(client,tokenizer):
        print('Actual isolated label IDs:',{s:tokenizer.encode(s,add_special_tokens=False) for s in LABELS})
        results=[]
        for policy in CANDIDATES:
            probe=task_namespace['live_task_probe'](client,tokenizer,CASES,
                output_cap=policy['output_cap'],structured=policy['structured'],system=policy['system'],
                save_partial=lambda result:lab_namespace['checkpoint'](ATTEMPT/(policy['name']+'.partial.json'),result))
            results.append(dict(policy=policy,probe=probe,summary=task_summary(probe['observations'])))
        return dict(status='complete',split='calibration',results=results)
    ATTEMPT=Path('evidence')/datetime.now(timezone.utc).strftime('topic04-%Y%m%dT%H%M%S%fZ')
    ATTEMPT.mkdir(parents=True,exist_ok=False)
    RUN=lab_namespace['run_lab'](MODEL_DIR,ATTEMPT/'calibration-service.json',task_probe=task_probe)
    assert lab_namespace['validate_artifact'](RUN)
    FRESH_CLIENT=assemble_response(decode_chunks([bytes.fromhex(c['hex']) for c in RUN['raw_sse']['chunks']]))
    assert FRESH_CLIENT['content']==RUN['raw_sse']['content'] and FRESH_CLIENT['status']=='complete'
    CALIBRATION=RUN['task_probe']
    assert all(len(r['probe']['observations'])==12 for r in CALIBRATION['results'])
    for result in CALIBRATION['results']:
        print(result['policy']['name'],result['summary'])
        plot_matrix(result['probe']['observations'],score_predictions,'Fresh vLLM / calibration / '+result['policy']['name'])
    print('Teardown:',RUN['teardown'])
else: print('Not run: no fresh vLLM task/T4 claim')


## D3. Choose from the observed calibration results

Fill CHOSEN_POLICY with an observed name. Explain accuracy, validity and at least one class recall, compared with the baseline. A tie can justify keeping the simpler original configuration. A format constraint can improve syntax without improving semantics. No guaranteed winner is supplied.

Saving requires completed, matched calibration observations and both declared policies. The decision file is immutable within this attempt. A different choice requires a new attempt. Held-out is not used to choose it. This checkpoint checks integrity, not the quality of your explanation.


In [ ]:
CHOSEN_POLICY = ""
SELECTION_EXPLANATION = ""
if RUN_GPU:
    FROZEN=task_namespace['freeze_selection'](CALIBRATION,CANDIDATES,CHOSEN_POLICY,
        SELECTION_EXPLANATION,ATTEMPT/'decision.json')
    print('Frozen choice:',FROZEN['payload']['selected'],'SHA256:',FROZEN['sha256'])
else: print('GPU disabled: no measured choice fabricated')


## E. Open held-out, test once, then write the memo

Only now load six new public teaching tickets. The loader requires the saved choice. Their labels remain inspectable in provided source; this is a teaching checkpoint, not a secret blind test. Restart the same cached FP16 model with the same server configuration; evaluate only the chosen policy, then stop it. No retuning from this score. Startup/restart times are excluded from quality and are not a speed comparison.


In [ ]:
if RUN_GPU:
    FROZEN=task_namespace['read_frozen'](ATTEMPT/'decision.json')
    HELDOUT=task_namespace['heldout_cases'](FROZEN)
    selected=FROZEN['payload']['selected']
    def heldout_probe(client,tokenizer):
        return task_namespace['live_task_probe'](client,tokenizer,HELDOUT,
            output_cap=selected['output_cap'],structured=selected['structured'],system=selected['system'],
            save_partial=lambda result:lab_namespace['checkpoint'](ATTEMPT/'heldout.partial.json',result))
    TEST_RUN=lab_namespace['run_lab'](MODEL_DIR,ATTEMPT/'heldout-service.json',task_probe=heldout_probe)
    assert lab_namespace['validate_artifact'](TEST_RUN)
    for key in ('model_id','revision','served_name','vllm','torch','cuda','transformers','launch_args'):
        assert RUN[key]==TEST_RUN[key],('Runtime changed between calibration and test',key)
    TEST=TEST_RUN['task_probe']
    assert TEST['status']=='complete' and len(TEST['observations'])==6
    assert task_namespace['read_frozen'](ATTEMPT/'decision.json')==FROZEN
    print('Held-out:',task_summary(TEST['observations']),'Teardown:',TEST_RUN['teardown'])
    plot_matrix(TEST['observations'],score_predictions,'Fresh vLLM / held-out / '+selected['name'])
    Path('evidence/topic-04-service.json').write_text(json.dumps(TEST_RUN,indent=2,ensure_ascii=False))
    Path('evidence/topic-04-task.json').write_text(json.dumps(dict(calibration=CALIBRATION,
        frozen=FROZEN,heldout=TEST,heldout_summary=task_summary(TEST['observations'])),indent=2,ensure_ascii=False))
else: print('Held-out remains unopened; no GPU evidence')


## F. Memo and transfer

Keep notebook, topic-04-client.json, fresh validated topic-04-service.json, topic-04-task.json and topic-04-analysis.md. CPU/task files supplement the unchanged graded service artifact.

Report task/model revision; positional budget; declared one-factor hypothesis; both calibration matrices, constant-access baseline and per-class recall; your frozen choice and held-out result; two actual errors and a next controlled test. Retain decision.json and both service captures inside the attempt folder. Separate transport, format and semantics. This is a task-quality comparison, not a speed A/B. Structured choice does not guarantee accuracy.

**Transfer:** seven new attempts have four complete correct labels, one complete wrong label, one complete non-label and one partial correct-looking prefix. Compute all-attempt accuracy, valid rate and completion rate. Explain why valid-only accuracy differs. Which observation would justify changing GPU flags?

Optional: write an SSE reducer or investigate buffering/cancellation. Technical rehearsal is not independent learner validation.
